# Fair Drop — Person 2 Final ML Notebook

This notebook is the **Person 2 workbench**. It uses the original `bot-detection_01` implementation as the engineering reference (feature extraction, windowing, coordination engine and risk fusion) and adds the larger 67,352-session dataset for the main web-behavior model.

**Outputs:**
1. Model 1 — large-data behavioral bot classifier
2. Model 2 — human-normal anomaly detector (Isolation Forest)
3. Campaign/coordination intelligence using the repo's actual coordination engine
4. Fair Drop adversarial simulator/evaluation
5. Versioned RiskEvent artifacts

Run all cells from top to bottom. No terminal commands are required.

In [1]:
from pathlib import Path
import sys, json, time, random, math
import numpy as np
import pandas as pd

# Locate the packaged repo even when this notebook is opened from /notebooks.
HERE = Path.cwd().resolve()
CANDIDATES = [HERE, *HERE.parents]
ROOT = next((p for p in CANDIDATES if (p / "repo" / "detection" / "coordination_engine.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate packaged repo folder.")
REPO = ROOT / "repo"
EXTERNAL = ROOT / "external_data"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
import joblib

from processing.feature_engine import build_feature_dataframe, build_window_feature_dataframe
from detection.coordination_engine import (
    select_candidate_sessions, build_similarity_matrix,
    find_suspicious_pairs, cluster_pairs
)
from detection.risk_fusion import fuse_risk, build_alert

print("Person 2 environment ready")
print("Repo:", REPO)

Person 2 environment ready
Repo: C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\repo


## 1. Load the large primary dataset

This is the Zenodo `simple_features.csv` supplied for the project: **67,352 labelled sessions**. `ID` is treated as an identifier and is **not** used as a feature.

In [2]:
primary = pd.read_csv(EXTERNAL / "simple_features.csv")
print("Shape:", primary.shape)
print("Label counts:\n", primary["ROBOT"].value_counts())
print("Missing values:", int(primary.isna().sum().sum()))

Shape: (67352, 32)
Label counts:
 ROBOT
0    53858
1    13494
Name: count, dtype: int64
Missing values: 43221


## 2. Model 1 — large-data behavioral bot classifier

This follows the repo's **Random Forest session-model approach**, but trains on the much larger labelled session dataset. The split is held out before model selection. We do not use the session ID.

In [3]:
DROP = ["ID", "ROBOT"]
X = primary.drop(columns=DROP).copy()
y = primary["ROBOT"].astype(int)
X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model1 = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model1.fit(X_train, y_train)
model1_prob = model1.predict_proba(X_test)[:, 1]
model1_pred = (model1_prob >= 0.50).astype(int)

m1_metrics = {
    "accuracy": accuracy_score(y_test, model1_pred),
    "precision": precision_score(y_test, model1_pred),
    "recall": recall_score(y_test, model1_pred),
    "f1": f1_score(y_test, model1_pred),
    "roc_auc": roc_auc_score(y_test, model1_prob),
    "pr_auc": average_precision_score(y_test, model1_prob),
    "test_rows": len(y_test),
}
print(json.dumps(m1_metrics, indent=2))
print(classification_report(y_test, model1_pred, target_names=["human", "bot"]))

{
  "accuracy": 0.9792888426991314,
  "precision": 0.9425749817117776,
  "recall": 0.9547980733605039,
  "f1": 0.9486471562672557,
  "roc_auc": 0.9954790815924315,
  "pr_auc": 0.9811318737717991,
  "test_rows": 13471
}
              precision    recall  f1-score   support

       human       0.99      0.99      0.99     10772
         bot       0.94      0.95      0.95      2699

    accuracy                           0.98     13471
   macro avg       0.97      0.97      0.97     13471
weighted avg       0.98      0.98      0.98     13471



### Model 1 metric note
The notebook reports the **unseen holdout** honestly. It does not force an artificial >98% result by using IDs, training data, or leaked labels. If the supplied dataset's held-out accuracy is below 98%, that is the measured result.

In [4]:
# Save Model 1 + schema after evaluation.
joblib.dump(model1, ARTIFACTS / "model1_large_behavioral_random_forest.joblib")
(ARTIFACTS / "model1_large_feature_columns.json").write_text(
    json.dumps(list(X.columns), indent=2), encoding="utf-8"
)
(ARTIFACTS / "model1_large_metrics.json").write_text(
    json.dumps(m1_metrics, indent=2), encoding="utf-8"
)
print("Saved Model 1 artifacts")

Saved Model 1 artifacts


## 3. Model 2 — behavioral anomaly detector

Model 2 answers a different question: **how unusual is this session compared with legitimate human sessions?** It is trained only on the human training population. The anomaly score is not a probability.

In [5]:
human_mask = y_train == 0
X_human_train = X_train.loc[human_mask]

model2 = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)
model2.fit(X_human_train)

# Higher = more unusual, matching the project's anomaly-score convention.
def anomaly_score(model, frame):
    raw = -model.score_samples(frame)
    lo, hi = float(raw.min()), float(raw.max())
    # Keep a bounded, monotonic 0..1 score for the RiskEvent contract.
    if hi == lo:
        return np.zeros(len(raw))
    return (raw - lo) / (hi - lo)

train_human_scores = anomaly_score(model2, X_human_train)
threshold = float(np.quantile(train_human_scores, 0.95))

test_scores = anomaly_score(model2, X_test)
human_test = y_test.to_numpy() == 0
bot_test = y_test.to_numpy() == 1
m2_pred = (test_scores >= threshold).astype(int)

m2_metrics = {
    "human_training_rows": int(len(X_human_train)),
    "threshold_95pct_human": threshold,
    "heldout_human_false_positive_rate": float(m2_pred[human_test].mean()),
    "heldout_bot_detection_rate": float(m2_pred[bot_test].mean()),
    "heldout_rows": int(len(X_test)),
}
print(json.dumps(m2_metrics, indent=2))

{
  "human_training_rows": 43086,
  "threshold_95pct_human": 0.4830743793999816,
  "heldout_human_false_positive_rate": 0.03369847753434831,
  "heldout_bot_detection_rate": 0.21563542052612078,
  "heldout_rows": 13471
}


In [6]:
joblib.dump(model2, ARTIFACTS / "model2_human_anomaly_isolation_forest.joblib")
(ARTIFACTS / "model2_metadata.json").write_text(
    json.dumps({"threshold": threshold, **m2_metrics}, indent=2), encoding="utf-8"
)
print("Saved Model 2 artifacts")

Saved Model 2 artifacts


## 4. Use the repo's actual feature extraction + coordination code

This section imports the original repository modules rather than replacing them. We regenerate session/window features from its `data/events.csv`, then pass those windows into its `coordination_engine.py`.

In [7]:
# Re-run the repository's own training entry points so the original implementation is preserved as a reference artifact.
from models.train_model import main as train_repo_session_model
from models.train_window_model import main as train_repo_window_model

print("Training original repo session model...")
train_repo_session_model()
print("Training original repo window model...")
train_repo_window_model()


Training original repo session model...
=== Logistic Regression Accuracy ===
0.9286
Predictions: ['bot', 'bot', 'bot', 'human', 'human', 'bot', 'bot', 'bot', 'human', 'human', 'bot', 'human', 'bot', 'bot']
Actual: ['bot', 'bot', 'bot', 'human', 'human', 'bot', 'bot', 'human', 'human', 'human', 'bot', 'human', 'bot', 'bot']
=== Random Forest Accuracy ===
0.8571
Predictions: ['bot', 'bot', 'bot', 'bot', 'human', 'bot', 'bot', 'human', 'human', 'human', 'bot', 'bot', 'bot', 'bot']
Actual: ['bot', 'bot', 'bot', 'human', 'human', 'bot', 'bot', 'human', 'human', 'human', 'bot', 'human', 'bot', 'bot']
Saved feature importance plot to C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\repo\models\feature_importance.png
=== Feature Importance ===
keydown_ratio       0.113289
keydown_count       0.092485
bigram_KK           0.091280
mousemove_ratio     0.057242
bigram_SS           0.053898
event_rate          0.051278
session_duration    0.050003
longest_idle_gap    0.

C:\Users\Sohana Pilli\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:200: UserWarning: [01:59:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost Accuracy ===
0.8571
=== Model Comparison ===
Logistic Regression: 0.9286
Random Forest: 0.8571
XGBoost: 0.8571
Saved session metrics to C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\repo\models\session_metrics.json
Training original repo window model...
Held-out sessions: ['ffd821a6-07ba-4dd2-99dc-3281a8e368fb', 'fe899d06-6d74-4e9f-8b3d-acbb4b408694']
=== Window Logistic Regression Accuracy ===
0.9756
Predictions: ['human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'bot', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'bot', 'bot', 'bot', 'bot']
Actual: ['human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human', 'human'

In [8]:
repo_events = pd.read_csv(REPO / "data" / "events.csv")
repo_events["timestamp"] = pd.to_numeric(repo_events["timestamp"], errors="coerce")
repo_events = repo_events.dropna(subset=["session_id", "event_type", "timestamp"]).copy()
repo_session_features = build_feature_dataframe(repo_events)
repo_windows = build_window_feature_dataframe(repo_events)

print("Repo raw events:", repo_events.shape)
print("Repo session features:", repo_session_features.shape)
print("Repo windows:", repo_windows.shape)
print("Session labels:\n", repo_session_features["label"].value_counts())

Repo raw events: (10180, 7)
Repo session features: (71, 36)
Repo windows: (518, 39)
Session labels:
 label
bot      39
human    32
Name: count, dtype: int64


In [9]:
# Run the original repository coordination functions.
candidates = select_candidate_sessions(repo_windows)
sim_matrix = build_similarity_matrix(candidates)
pairs = find_suspicious_pairs(candidates, sim_matrix, similarity_threshold=0.90, time_threshold=5.0)
groups = cluster_pairs(pairs)

campaign_summary = {
    "candidate_bot_windows": int(len(candidates)),
    "suspicious_pairs": int(len(pairs)),
    "campaign_groups": int(sum(1 for g in groups if len(set(candidates.iloc[list(g)]["session_id"].tolist())) >= 2)),
    "largest_campaign_size": int(max((len(g) for g in groups), default=0)),
}
print(json.dumps(campaign_summary, indent=2))

{
  "candidate_bot_windows": 168,
  "suspicious_pairs": 441,
  "campaign_groups": 9,
  "largest_campaign_size": 28
}


## 5. Campaign intelligence

A campaign is formed from suspicious sessions/windows that are both behaviorally similar and temporally close. This is the repo's coordination mechanism, mapped to Fair Drop's campaign/identity layer.

In [10]:
campaign_rows = []
for cid, group in enumerate(groups, start=1):
    sessions = sorted(set(candidates.iloc[list(group)]["session_id"].tolist()))
    if len(sessions) < 2:
        continue
    campaign_rows.append({
        "campaign_id": f"camp_{cid:03d}",
        "session_ids": sessions,
        "size": len(sessions),
    })

campaign_df = pd.DataFrame(campaign_rows)
if not campaign_df.empty:
    print(campaign_df.to_string(index=False))
else:
    print("No multi-session campaigns found under the repo thresholds.")

campaign_id                                                                                                                                                                                                                                                                                                                                            session_ids  size
   camp_002 [1bed4691-cf7f-4424-a723-d09e9d7c41bb, 2f2a5540-fceb-4e5d-a52e-c150885a4c2d, 4c5357a4-d1d9-47dd-8b90-5b36ef405048, 5641770d-75ee-4361-8772-69eba8aacd96, 65e023b1-f22f-4b9f-8788-8f873bd0c7ac, 923b58c7-1602-4f98-8413-ae6c1a3960aa, bb83cc53-c962-4802-be6c-e4c2e11eb094, f4b8657a-27d1-4afb-b8c9-a5045f9bbd16, ffd821a6-07ba-4dd2-99dc-3281a8e368fb]     9
   camp_006                                                                                                                                                                                                                                     [2f2a5540-fceb-4e5d-a52e-c150885a4c2d,

## 6. Fair Drop adversarial simulator (offline)

The simulator follows the **behavioral idea and timing profiles of the repo's `synthetic_runner.py`**, but does not require Flask/Selenium. It produces reproducible event streams for human, fast, low-and-slow, refresh-flood, queue-jump and coordinated scenarios.

In [11]:
EVENTS = ["page_screen_change", "focus_change", "mousemove", "click", "keydown", "scroll"]

PROFILE = {
    "human": {"nav": (250, 600), "move": (80, 220), "type": (70, 160), "click": (120, 260), "scroll": (180, 350)},
    "fast": {"nav": (60, 120), "move": (20, 35), "type": (12, 24), "click": (18, 40), "scroll": (30, 45)},
    "low_and_slow": {"nav": (450, 900), "move": (120, 260), "type": (80, 180), "click": (160, 320), "scroll": (220, 420)},
    "refresh_flood": {"nav": (35, 80), "move": (15, 30), "type": (10, 20), "click": (12, 25), "scroll": (20, 35)},
    "queue_jump": {"nav": (45, 100), "move": (15, 30), "type": (10, 20), "click": (15, 30), "scroll": (25, 40)},
    "coordinated": {"nav": (90, 140), "move": (35, 60), "type": (18, 32), "click": (28, 50), "scroll": (45, 70)},
}

def simulate_session(session_id, profile, start_ms, seed):
    rng = random.Random(seed)
    p = PROFILE[profile]
    t = start_ms
    rows = []
    def emit(event_type, lo, hi):
        nonlocal t
        t += rng.randint(*p[lo]) if isinstance(lo, str) else rng.randint(lo, hi)
        rows.append({"session_id": session_id, "actor_type": "human" if profile == "human" else "bot", "bot_type": "none" if profile == "human" else profile, "event_type": event_type, "x": rng.randint(100, 1200), "y": rng.randint(100, 900), "timestamp": t})
    # Same journey shape as the repo's synthetic flow.
    for _ in range(2): emit("page_screen_change", *p["nav"])
    for _ in range(5): emit("mousemove", *p["move"])
    for _ in range(2): emit("click", *p["click"])
    for _ in range(12): emit("keydown", *p["type"])
    for _ in range(3): emit("scroll", *p["scroll"])
    for _ in range(3): emit("click", *p["click"])
    for _ in range(3): emit("page_screen_change", *p["nav"])
    # Fair Drop-specific attack actions.
    if profile == "refresh_flood":
        for _ in range(25): emit("page_screen_change", *p["nav"])
    elif profile == "queue_jump":
        for _ in range(12): emit("click", *p["click"])
    elif profile == "low_and_slow":
        for _ in range(5): emit("focus_change", *p["nav"])
    return rows

sim_rows=[]
base=1770000000000
profiles=["human","fast","low_and_slow","refresh_flood","queue_jump"]
for i in range(25):
    prof=profiles[i%len(profiles)]
    sim_rows.extend(simulate_session(f"sim_{i:03d}",prof,base+i*20000,100+i))
# coordinated campaign: synchronized start times, same behavior profile.
for i in range(5):
    sim_rows.extend(simulate_session(f"campaign_A_{i}","coordinated",base+500000+i*180,900+i))

sim_events=pd.DataFrame(sim_rows)
print("Simulated events:", sim_events.shape)
print(sim_events.groupby(["actor_type","bot_type"]).size())

Simulated events: (1110, 7)
actor_type  bot_type     
bot         coordinated      150
            fast             150
            low_and_slow     175
            queue_jump       210
            refresh_flood    275
human       none             150
dtype: int64


In [12]:
# Run the repo feature extractor on the new simulator output.
sim_sessions = build_feature_dataframe(sim_events)
sim_windows = build_window_feature_dataframe(sim_events)
print("Simulated session features:", sim_sessions.shape)
print("Simulated windows:", sim_windows.shape)

# Run the actual repo coordination engine on simulated windows.
sim_candidates = select_candidate_sessions(sim_windows)
sim_matrix = build_similarity_matrix(sim_candidates)
sim_pairs = find_suspicious_pairs(sim_candidates, sim_matrix, similarity_threshold=0.90, time_threshold=5.0)
sim_groups = cluster_pairs(sim_pairs)
print("Sim campaign pairs:", len(sim_pairs), "groups:", len(sim_groups))

Simulated session features: (30, 36)
Simulated windows: (110, 39)
Sim campaign pairs: 170 groups: 26


## 7. RiskEvent generation

The final object is designed to match Fair Drop's shared `RiskEvent` contract: session, risk, anomaly, coordination, campaign, attack type, evidence, model version and timestamp.

In [13]:
# End-to-end RiskEvent demo using the repository's actual session model + a session-level anomaly model + campaign score.
from sklearn.ensemble import IsolationForest

repo_model = joblib.load(REPO / "models" / "bot_model.pkl")
repo_drop = ["session_id", "actor_type", "bot_type", "label", "start_time"]
repo_X = repo_session_features.drop(columns=repo_drop).fillna(0.0)
repo_human = repo_X[repo_session_features["label"] == "human"]
repo_anomaly = IsolationForest(n_estimators=200, random_state=42).fit(repo_human)
repo_raw = -repo_anomaly.score_samples(repo_human)
repo_threshold = float(np.quantile(repo_raw, 0.95))

# Campaign membership map from multi-session groups only.
coord_by_session = {}
for cid, group in enumerate(groups, start=1):
    session_set = sorted(set(candidates.iloc[list(group)]["session_id"].tolist()))
    if len(session_set) < 2:
        continue
    campaign_id = f"sim_camp_{cid:03d}"
    for sid in session_set:
        coord_by_session[sid] = (campaign_id, 0.90)

risk_events=[]
for _, row in sim_sessions.iterrows():
    sid=row["session_id"]
    feature_row=row.drop(labels=[c for c in ["session_id","actor_type","bot_type","label","start_time"] if c in row.index]).to_frame().T.fillna(0.0)
    feature_row=feature_row.reindex(columns=repo_X.columns, fill_value=0.0)
    bot_score=float(repo_model.predict_proba(feature_row)[0,1])
    raw=-repo_anomaly.score_samples(feature_row)[0]
    # convert anomaly raw score into a bounded relative score using the human reference range
    anomaly_score=float(np.clip((raw-repo_raw.min())/(repo_raw.max()-repo_raw.min()+1e-12),0,1))
    campaign_id, coord = coord_by_session.get(sid, (None, 0.0))
    reasons=[]
    if bot_score >= 0.5: reasons.append("behavioral_bot_signal")
    if anomaly_score >= repo_threshold: reasons.append("unusual_behavior")
    if coord > 0: reasons.append("coordinated_behavior")
    final=float(fuse_risk(0.0, bot_score, coord))
    risk_events.append({
        "session_id": sid,
        "event_id": f"evt_{sid}",
        "risk_score": final,
        "anomaly_score": anomaly_score,
        "coordination_score": float(coord),
        "campaign_id": campaign_id,
        "attack_type": row["bot_type"],
        "evidence": reasons,
        "model_version": "fairdrop-person2-v1.0.0",
        "timestamp": int(time.time()*1000),
    })

risk_df=pd.DataFrame(risk_events)
risk_df.to_json(ARTIFACTS/"risk_events_simulator.json", orient="records", indent=2)
print(risk_df.head(12).to_string(index=False))

  session_id         event_id  risk_score  anomaly_score  coordination_score campaign_id   attack_type           evidence           model_version     timestamp
campaign_A_0 evt_campaign_A_0    0.107633       0.795061                 0.0        None   coordinated [unusual_behavior] fairdrop-person2-v1.0.0 1791059400147
campaign_A_1 evt_campaign_A_1    0.107633       0.792382                 0.0        None   coordinated [unusual_behavior] fairdrop-person2-v1.0.0 1791059400319
campaign_A_2 evt_campaign_A_2    0.107633       0.792382                 0.0        None   coordinated [unusual_behavior] fairdrop-person2-v1.0.0 1791059400473
campaign_A_3 evt_campaign_A_3    0.107633       0.792382                 0.0        None   coordinated [unusual_behavior] fairdrop-person2-v1.0.0 1791059400598
campaign_A_4 evt_campaign_A_4    0.107633       0.796402                 0.0        None   coordinated [unusual_behavior] fairdrop-person2-v1.0.0 1791059400723
     sim_000      evt_sim_000    0.14356

## 8. Save campaign/evaluation artifacts

In [14]:
campaign_df.to_json(ARTIFACTS/"campaigns_repo.json", orient="records", indent=2)
pd.DataFrame([m1_metrics]).to_csv(ARTIFACTS/"model1_metrics.csv", index=False)
pd.DataFrame([m2_metrics]).to_csv(ARTIFACTS/"model2_metrics.csv", index=False)

summary = {
    "primary_dataset_rows": int(len(primary)),
    "primary_human_rows": int((primary.ROBOT==0).sum()),
    "primary_bot_rows": int((primary.ROBOT==1).sum()),
    "model1": m1_metrics,
    "model2": m2_metrics,
    "repo_campaign": campaign_summary,
    "simulated_sessions": int(sim_sessions["session_id"].nunique()),
    "simulated_events": int(len(sim_events)),
    "simulated_campaign_groups": int(len(sim_groups)),
}
(ARTIFACTS/"person2_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))

{
  "primary_dataset_rows": 67352,
  "primary_human_rows": 53858,
  "primary_bot_rows": 13494,
  "model1": {
    "accuracy": 0.9792888426991314,
    "precision": 0.9425749817117776,
    "recall": 0.9547980733605039,
    "f1": 0.9486471562672557,
    "roc_auc": 0.9954790815924315,
    "pr_auc": 0.9811318737717991,
    "test_rows": 13471
  },
  "model2": {
    "human_training_rows": 43086,
    "threshold_95pct_human": 0.4830743793999816,
    "heldout_human_false_positive_rate": 0.03369847753434831,
    "heldout_bot_detection_rate": 0.21563542052612078,
    "heldout_rows": 13471
  },
  "repo_campaign": {
    "candidate_bot_windows": 168,
    "suspicious_pairs": 441,
    "campaign_groups": 9,
    "largest_campaign_size": 28
  },
  "simulated_sessions": 30,
  "simulated_events": 1110,
  "simulated_campaign_groups": 26
}


## Final interpretation

- **Model 1:** supervised behavioral bot classifier trained on the 67,352-session dataset.
- **Model 2:** Isolation Forest trained only on legitimate humans; it reports unusualness, not bot probability.
- **Campaign Intelligence:** the original repo coordination engine is reused and applied to event/window behavior; synchronized, highly similar sessions are grouped into campaigns.
- **Adversarial simulator:** follows the repo's synthetic behavior design and adds Fair Drop scenarios such as low-and-slow, refresh flood, queue jump and synchronized campaigns.

**Important:** do not claim that every component has >98% accuracy. Accuracy is meaningful for Model 1; Model 2 and campaign detection require different metrics. The notebook prints the measured unseen results instead of fabricating a target score.

In [15]:
# STEP 1 — Inspect the dataset we will mimic

import pandas as pd
import os

DATA_PATH = "../external_data/simple_features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(i, "->", col)

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (67352, 32)

Columns:
1 -> ID
2 -> NUMBER_OF_REQUESTS
3 -> TOTAL_DURATION
4 -> AVERAGE_TIME
5 -> STANDARD_DEVIATION
6 -> REPEATED_REQUESTS
7 -> HTTP_RESPONSE_2XX
8 -> HTTP_RESPONSE_3XX
9 -> HTTP_RESPONSE_4XX
10 -> HTTP_RESPONSE_5XX
11 -> GET_METHOD
12 -> POST_METHOD
13 -> HEAD_METHOD
14 -> OTHER_METHOD
15 -> NIGHT
16 -> UNASSIGNED
17 -> IMAGES
18 -> TOTAL_HTML
19 -> HTML_TO_IMAGE
20 -> HTML_TO_CSS
21 -> HTML_TO_JS
22 -> WIDTH
23 -> DEPTH
24 -> STD_DEPTH
25 -> CONSECUTIVE
26 -> DATA
27 -> PPI
28 -> SF_REFERRER
29 -> SF_FILETYPE
30 -> MAX_BARRAGE
31 -> PENALTY
32 -> ROBOT

First 5 rows:


,ID,NUMBER_OF_REQUESTS,TOTAL_DURATION,AVERAGE_TIME,STANDARD_DEVIATION,REPEATED_REQUESTS,HTTP_RESPONSE_2XX,HTTP_RESPONSE_3XX,HTTP_RESPONSE_4XX,HTTP_RESPONSE_5XX,...,DEPTH,STD_DEPTH,CONSECUTIVE,DATA,PPI,SF_REFERRER,SF_FILETYPE,MAX_BARRAGE,PENALTY,ROBOT
0,obSnwGoBCue8G08E_WCX,79,592,7.589744,1.800540,0.000000,0.873418,0.126582,0.000000,0.000000,...,4.0,0.494041,0.101266,1555089.0,2.718334e+07,0.000000,0.205128,1,0,1
1,nLSnwGoBCue8G08E_WCX,350,103621,296.908300,260.332670,0.000000,0.997143,0.000000,0.002857,0.000000,...,6.0,0.341303,0.425714,2071344.0,1.427714e+01,0.547278,0.461318,4,0,1
2,ErSowGoBCue8G08ECmHa,15,368,26.285715,87.329780,0.000000,1.000000,0.000000,0.000000,0.000000,...,3.0,0.000000,0.000000,49505.0,1.413333e+01,0.642857,1.000000,2,0,0
3,E7SowGoBCue8G08ECmHa,28,11,0.407407,1.097447,0.000000,1.000000,0.000000,0.000000,0.000000,...,3.0,0.000000,0.821429,45984.0,1.332143e+01,0.037037,0.148148,5,0,0
4,FbSowGoBCue8G08ECmHa,9342,80946,8.665668,17.566462,0.000107,0.996896,0.002569,0.000214,0.000321,...,4.0,0.152943,0.953757,83171793.0,1.307407e+01,0.000000,0.023552,1,0,1


In [16]:
# STEP 2 — Understand real Human vs Bot behaviour

print("ROBOT label counts:")
print(df["ROBOT"].value_counts())

print("\n0 = HUMAN")
print("1 = BOT")

# Pick important behavioural features
important_features = [
    "NUMBER_OF_REQUESTS",
    "TOTAL_DURATION",
    "AVERAGE_TIME",
    "STANDARD_DEVIATION",
    "REPEATED_REQUESTS",
    "GET_METHOD",
    "POST_METHOD",
    "HTTP_RESPONSE_2XX",
    "HTTP_RESPONSE_4XX",
    "CONSECUTIVE",
    "MAX_BARRAGE"
]

comparison = (
    df.groupby("ROBOT")[important_features]
      .median()
      .T
)

comparison.columns = ["HUMAN_MEDIAN", "BOT_MEDIAN"]

comparison["DIFFERENCE"] = (
    comparison["BOT_MEDIAN"] - comparison["HUMAN_MEDIAN"]
)

display(comparison)

ROBOT label counts:
ROBOT
0    53858
1    13494
Name: count, dtype: int64

0 = HUMAN
1 = BOT


,HUMAN_MEDIAN,BOT_MEDIAN,DIFFERENCE
NUMBER_OF_REQUESTS,30.000000,1.000000,-29.000000
TOTAL_DURATION,58.000000,0.000000,-58.000000
AVERAGE_TIME,1.922015,0.000000,-1.922015
STANDARD_DEVIATION,6.465380,2.827352,-3.638028
REPEATED_REQUESTS,0.013333,0.000000,-0.013333
GET_METHOD,1.000000,1.000000,0.000000
POST_METHOD,0.000000,0.000000,0.000000
HTTP_RESPONSE_2XX,1.000000,1.000000,0.000000
HTTP_RESPONSE_4XX,0.000000,0.000000,0.000000
CONSECUTIVE,0.428571,0.000000,-0.428571


In [17]:
# STEP 3 — Look at the behaviour ranges, not just the median

features_to_check = [
    "NUMBER_OF_REQUESTS",
    "TOTAL_DURATION",
    "AVERAGE_TIME",
    "STANDARD_DEVIATION",
    "REPEATED_REQUESTS",
    "CONSECUTIVE",
    "DEPTH",
    "MAX_BARRAGE"
]

for feature in features_to_check:
    print(f"\n===== {feature} =====")

    stats = df.groupby("ROBOT")[feature].quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    ).unstack()

    stats.index = ["HUMAN (0)", "BOT (1)"]

    display(stats)


===== NUMBER_OF_REQUESTS =====


,0.10,0.25,0.50,0.75,0.90
HUMAN (0),1.0,7.0,30.0,75.0,162.0
BOT (1),1.0,1.0,1.0,3.0,20.0



===== TOTAL_DURATION =====


,0.10,0.25,0.50,0.75,0.90
HUMAN (0),0.0,4.0,58.0,372.00,1311.0
BOT (1),0.0,0.0,0.0,155.75,2664.3



===== AVERAGE_TIME =====


...

,0.10,0.25,0.50,0.75,0.90
HUMAN (0),0.0,0.24,0.428571,0.607786,0.731604
BOT (1),0.0,0.00,0.000000,0.328697,0.612753



===== DEPTH =====


,0.10,0.25,0.50,0.75,0.90
HUMAN (0),3.0,4.0,6.0,6.0,6.0
BOT (1),3.0,3.0,3.0,4.0,6.0



===== MAX_BARRAGE =====


,0.10,0.25,0.50,0.75,0.90
HUMAN (0),0.0,0.0,14.0,30.0,48.0
BOT (1),0.0,0.0,0.0,0.0,1.0


In [18]:
# STEP 4 — Build a dataset-like behavioral simulator

import numpy as np
import pandas as pd

# Columns used to generate behaviour
SIM_FEATURES = [
    "NUMBER_OF_REQUESTS",
    "TOTAL_DURATION",
    "AVERAGE_TIME",
    "STANDARD_DEVIATION",
    "REPEATED_REQUESTS",
    "HTTP_RESPONSE_2XX",
    "HTTP_RESPONSE_3XX",
    "HTTP_RESPONSE_4XX",
    "HTTP_RESPONSE_5XX",
    "GET_METHOD",
    "POST_METHOD",
    "HEAD_METHOD",
    "OTHER_METHOD",
    "NIGHT",
    "UNASSIGNED",
    "IMAGES",
    "TOTAL_HTML",
    "HTML_TO_IMAGE",
    "HTML_TO_CSS",
    "HTML_TO_JS",
    "WIDTH",
    "DEPTH",
    "STD_DEPTH",
    "CONSECUTIVE",
    "DATA",
    "PPI",
    "SF_REFERRER",
    "SF_FILETYPE",
    "MAX_BARRAGE",
    "PENALTY"
]

def generate_simulated_sessions(
    source_df,
    robot_label,
    n_sessions=10,
    noise=0.08,
    seed=42
):
    """
    Generate unseen sessions that preserve the statistical
    behaviour of the original dataset.

    robot_label:
        0 = human
        1 = bot
    """

    rng = np.random.default_rng(seed)

    # Take only the requested real behaviour class
    source = source_df[source_df["ROBOT"] == robot_label].copy()

    # Randomly select source behaviour
    sampled = source.sample(
        n=n_sessions,
        replace=True,
        random_state=seed
    ).copy()

    simulated = sampled[SIM_FEATURES].copy()

    # Perturb numeric behaviour slightly
    for col in SIM_FEATURES:

        values = pd.to_numeric(
            simulated[col],
            errors="coerce"
        )

        # Replace missing values with class median
        class_median = pd.to_numeric(
            source[col],
            errors="coerce"
        ).median()

        values = values.fillna(class_median)

        # Add controlled noise
        if values.std() > 0:
            noise_values = rng.normal(
                loc=0,
                scale=values.std() * noise,
                size=len(values)
            )

            values = values + noise_values

        # Counts cannot be negative
        if col in [
            "NUMBER_OF_REQUESTS",
            "TOTAL_DURATION",
            "IMAGES",
            "TOTAL_HTML",
            "WIDTH",
            "DEPTH",
            "DATA",
            "MAX_BARRAGE"
        ]:
            values = values.clip(lower=0)

        simulated[col] = values

    # Add simulator metadata
    simulated.insert(
        0,
        "SIM_SESSION_ID",
        [f"sim_{robot_label}_{i:04d}" for i in range(n_sessions)]
    )

    simulated["TRUE_LABEL"] = robot_label

    return simulated


# Generate a small first batch
sim_humans = generate_simulated_sessions(
    df,
    robot_label=0,
    n_sessions=10,
    noise=0.08,
    seed=42
)

sim_bots = generate_simulated_sessions(
    df,
    robot_label=1,
    n_sessions=10,
    noise=0.08,
    seed=99
)

simulator_data = pd.concat(
    [sim_humans, sim_bots],
    ignore_index=True
)

print("Simulator created!")
print("Shape:", simulator_data.shape)

print("\nTrue labels:")
print(simulator_data["TRUE_LABEL"].value_counts())

display(simulator_data.head())

Simulator created!
Shape: (20, 32)

True labels:
TRUE_LABEL
0    10
1    10
Name: count, dtype: int64


,SIM_SESSION_ID,NUMBER_OF_REQUESTS,TOTAL_DURATION,AVERAGE_TIME,STANDARD_DEVIATION,REPEATED_REQUESTS,HTTP_RESPONSE_2XX,HTTP_RESPONSE_3XX,HTTP_RESPONSE_4XX,HTTP_RESPONSE_5XX,...,DEPTH,STD_DEPTH,CONSECUTIVE,DATA,PPI,SF_REFERRER,SF_FILETYPE,MAX_BARRAGE,PENALTY,TRUE_LABEL
0,sim_0_0000,29.937196,98.501016,1.660234,15.867539,0.212357,1.001925,-0.011204,0.0,0.0,...,5.926753,1.214839,0.398442,170542.271989,9.445009,0.412910,0.531570,5.648077,3.855101,0
1,sim_0_0001,12.801396,1720.168131,110.557403,221.059404,0.253990,1.004203,-0.002229,0.0,0.0,...,5.944959,0.888141,0.253214,52174.745585,10.711422,0.570959,0.604490,0.050664,5.984423,0
2,sim_0_0002,75.308108,23.815811,3.719566,-1.403737,0.008810,0.990299,0.001084,0.0,0.0,...,6.161539,1.301739,0.637816,249718.211570,10.771054,0.125671,0.211603,29.507214,0.709540,0
3,sim_0_0003,4.892827,50.070033,1.566713,-4.499045,0.001705,0.997872,0.003903,0.0,0.0,...,2.937661,-0.041017,0.016206,24396.423916,14.125322,0.989399,0.979922,1.937948,-0.242425,0
4,sim_0_0004,23.999343,21.936452,-1.132028,3.658915,0.000857,0.996869,0.004735,0.0,0.0,...,6.063637,0.885089,0.372948,203546.810160,5.034332,0.165240,0.277537,0.000000,0.351852,0


In [21]:
# STEP 5 — Test Model 1 on simulated sessions

# Get the EXACT feature columns used when Model 1 was trained
model1_features = list(model1.feature_names_in_)

# Prepare simulated data using those exact columns
X_sim = simulator_data[model1_features].copy()

# Convert everything to numeric
X_sim = X_sim.apply(pd.to_numeric, errors="coerce")

# Fill missing values using the real dataset medians
X_sim = X_sim.fillna(
    df[model1_features].apply(
        pd.to_numeric,
        errors="coerce"
    ).median()
)

# Model 1 prediction
sim_predictions = model1.predict(X_sim)

# Get probability of BOT (class = 1)
bot_class_index = list(model1.classes_).index(1)
sim_bot_probability = model1.predict_proba(X_sim)[:, bot_class_index]

# Create results table
simulation_results = simulator_data[
    ["SIM_SESSION_ID", "TRUE_LABEL"]
].copy()

simulation_results["MODEL1_PREDICTION"] = sim_predictions
simulation_results["MODEL1_BOT_PROBABILITY"] = sim_bot_probability

# Human-readable labels
simulation_results["TRUE_TYPE"] = simulation_results["TRUE_LABEL"].map({
    0: "HUMAN",
    1: "BOT"
})

simulation_results["MODEL1_TYPE"] = simulation_results[
    "MODEL1_PREDICTION"
].map({
    0: "HUMAN",
    1: "BOT"
})

print("Model 1 tested on simulated sessions.")
print("Simulated sessions:", len(simulation_results))

display(simulation_results)

Model 1 tested on simulated sessions.
Simulated sessions: 20


,SIM_SESSION_ID,TRUE_LABEL,MODEL1_PREDICTION,MODEL1_BOT_PROBABILITY,TRUE_TYPE,MODEL1_TYPE
0,sim_0_0000,0,0,0.000000,HUMAN,HUMAN
1,sim_0_0001,0,0,0.010000,HUMAN,HUMAN
2,sim_0_0002,0,0,0.026000,HUMAN,HUMAN
3,sim_0_0003,0,0,0.030000,HUMAN,HUMAN
4,sim_0_0004,0,0,0.012000,HUMAN,HUMAN
5,sim_0_0005,0,0,0.018000,HUMAN,HUMAN
6,sim_0_0006,0,0,0.012000,HUMAN,HUMAN
7,sim_0_0007,0,0,0.006000,HUMAN,HUMAN
8,sim_0_0008,0,0,0.016000,HUMAN,HUMAN
9,sim_0_0009,0,0,0.000000,HUMAN,HUMAN


In [22]:
# STEP 6 — Evaluate Model 1 on simulated sessions

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = simulation_results["TRUE_LABEL"]
y_pred = simulation_results["MODEL1_PREDICTION"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("===== MODEL 1 — SIMULATOR RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== CONFUSION MATRIX =====")

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Human", "Actual Bot"],
    columns=["Predicted Human", "Predicted Bot"]
)

display(cm_df)

===== MODEL 1 — SIMULATOR RESULTS =====
Accuracy : 0.9000
Precision: 1.0000
Recall   : 0.8000
F1 Score : 0.8889

===== CONFUSION MATRIX =====


,Predicted Human,Predicted Bot
Actual Human,10,0
Actual Bot,2,8


In [24]:
# STEP 8 — Inspect Model 2's actual scoring calibration

print("Model 2 type:")
print(type(model2))

print("\nIsolation Forest attributes:")
for attr in dir(model2):
    if "feature" in attr.lower() or "offset" in attr.lower():
        print(attr)

print("\nModel 2 threshold:")
print("0.4830743793999816")

print("\nRaw anomaly score range on simulated data:")

model2_features = list(model2.feature_names_in_)

X_sim_m2 = simulator_data[model2_features].copy()

X_sim_m2 = X_sim_m2.apply(pd.to_numeric, errors="coerce")

m2_medians = (
    df[model2_features]
    .apply(pd.to_numeric, errors="coerce")
    .median()
)

X_sim_m2 = X_sim_m2.fillna(m2_medians)

raw_scores = -model2.score_samples(X_sim_m2)

print("Minimum raw score:", raw_scores.min())
print("Maximum raw score:", raw_scores.max())
print("Mean raw score:", raw_scores.mean())
print("Median raw score:", np.median(raw_scores))

Model 2 type:
<class 'sklearn.ensemble._iforest.IsolationForest'>

Isolation Forest attributes:
_max_features
bootstrap_features
estimators_features_
feature_names_in_
max_features
n_features_in_
offset_

Model 2 threshold:
0.4830743793999816

Raw anomaly score range on simulated data:
Minimum raw score: 0.3501672257289803
Maximum raw score: 0.5283041685235866
Mean raw score: 0.4234579238246086
Median raw score: 0.41816539086000404


In [25]:
# STEP 9 — Correct Model 2 test on simulated sessions

MODEL2_THRESHOLD = 0.4830743793999816

model2_features = list(model2.feature_names_in_)

X_sim_m2 = simulator_data[model2_features].copy()

X_sim_m2 = X_sim_m2.apply(pd.to_numeric, errors="coerce")

m2_medians = (
    df[model2_features]
    .apply(pd.to_numeric, errors="coerce")
    .median()
)

X_sim_m2 = X_sim_m2.fillna(m2_medians)

# IMPORTANT:
# Use the raw Isolation Forest score directly.
# Do NOT normalize the 20 simulator rows.
raw_anomaly_scores = -model2.score_samples(X_sim_m2)

simulation_results["MODEL2_ANOMALY_SCORE"] = raw_anomaly_scores

simulation_results["MODEL2_ANOMALOUS"] = (
    simulation_results["MODEL2_ANOMALY_SCORE"]
    >= MODEL2_THRESHOLD
)

display(
    simulation_results[
        [
            "SIM_SESSION_ID",
            "TRUE_TYPE",
            "MODEL1_TYPE",
            "MODEL1_BOT_PROBABILITY",
            "MODEL2_ANOMALY_SCORE",
            "MODEL2_ANOMALOUS"
        ]
    ]
)

,SIM_SESSION_ID,TRUE_TYPE,MODEL1_TYPE,MODEL1_BOT_PROBABILITY,MODEL2_ANOMALY_SCORE,MODEL2_ANOMALOUS
0,sim_0_0000,HUMAN,HUMAN,0.000000,0.384103,False
1,sim_0_0001,HUMAN,HUMAN,0.010000,0.457775,False
2,sim_0_0002,HUMAN,HUMAN,0.026000,0.417487,False
3,sim_0_0003,HUMAN,HUMAN,0.030000,0.422195,False
4,sim_0_0004,HUMAN,HUMAN,0.012000,0.398519,False
5,sim_0_0005,HUMAN,HUMAN,0.018000,0.358956,False
6,sim_0_0006,HUMAN,HUMAN,0.012000,0.407252,False
7,sim_0_0007,HUMAN,HUMAN,0.006000,0.350167,False
8,sim_0_0008,HUMAN,HUMAN,0.016000,0.455175,False
9,sim_0_0009,HUMAN,HUMAN,0.000000,0.377928,False


In [26]:
# STEP 10 — Model 2 threshold experiment

thresholds = np.arange(0.35, 0.531, 0.01)

threshold_results = []

y_true = simulation_results["TRUE_LABEL"].values
scores = simulation_results["MODEL2_ANOMALY_SCORE"].values

for threshold in thresholds:

    predicted_anomaly = (scores >= threshold).astype(int)

    human_mask = (y_true == 0)
    bot_mask = (y_true == 1)

    human_false_positive_rate = (
        predicted_anomaly[human_mask].mean()
    )

    bot_detection_rate = (
        predicted_anomaly[bot_mask].mean()
    )

    threshold_results.append({
        "threshold": round(threshold, 3),
        "human_FPR": human_false_positive_rate,
        "bot_detection_rate": bot_detection_rate
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results)

,threshold,human_FPR,bot_detection_rate
0,0.35,1.0,1.0
1,0.36,0.8,1.0
2,0.37,0.8,1.0
3,0.38,0.7,1.0
4,0.39,0.6,1.0
5,0.40,0.5,1.0
6,0.41,0.4,1.0
7,0.42,0.3,0.5
8,0.43,0.2,0.5
9,0.44,0.2,0.3


In [27]:
# STEP 10 — Model 2 threshold experiment

thresholds = np.arange(0.35, 0.531, 0.01)

threshold_results = []

y_true = simulation_results["TRUE_LABEL"].values
scores = simulation_results["MODEL2_ANOMALY_SCORE"].values

for threshold in thresholds:

    predicted_anomaly = (scores >= threshold).astype(int)

    human_mask = (y_true == 0)
    bot_mask = (y_true == 1)

    human_false_positive_rate = (
        predicted_anomaly[human_mask].mean()
    )

    bot_detection_rate = (
        predicted_anomaly[bot_mask].mean()
    )

    threshold_results.append({
        "threshold": round(threshold, 3),
        "human_FPR": human_false_positive_rate,
        "bot_detection_rate": bot_detection_rate
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results)

,threshold,human_FPR,bot_detection_rate
0,0.35,1.0,1.0
1,0.36,0.8,1.0
2,0.37,0.8,1.0
3,0.38,0.7,1.0
4,0.39,0.6,1.0
5,0.40,0.5,1.0
6,0.41,0.4,1.0
7,0.42,0.3,0.5
8,0.43,0.2,0.5
9,0.44,0.2,0.3


In [28]:
# STEP 11 — Build a realistic, valid simulator

import numpy as np
import pandas as pd

SIM_FEATURES = [
    "NUMBER_OF_REQUESTS", "TOTAL_DURATION", "AVERAGE_TIME",
    "STANDARD_DEVIATION", "REPEATED_REQUESTS",
    "HTTP_RESPONSE_2XX", "HTTP_RESPONSE_3XX", "HTTP_RESPONSE_4XX",
    "HTTP_RESPONSE_5XX", "GET_METHOD", "POST_METHOD", "HEAD_METHOD",
    "OTHER_METHOD", "NIGHT", "UNASSIGNED", "IMAGES", "TOTAL_HTML",
    "HTML_TO_IMAGE", "HTML_TO_CSS", "HTML_TO_JS", "WIDTH", "DEPTH",
    "STD_DEPTH", "CONSECUTIVE", "DATA", "PPI", "SF_REFERRER",
    "SF_FILETYPE", "MAX_BARRAGE", "PENALTY"
]

# Features that cannot logically be negative
NON_NEGATIVE = [
    "NUMBER_OF_REQUESTS", "TOTAL_DURATION", "AVERAGE_TIME",
    "STANDARD_DEVIATION", "HTTP_RESPONSE_2XX", "HTTP_RESPONSE_3XX",
    "HTTP_RESPONSE_4XX", "HTTP_RESPONSE_5XX", "GET_METHOD",
    "POST_METHOD", "HEAD_METHOD", "OTHER_METHOD", "IMAGES",
    "TOTAL_HTML", "WIDTH", "DEPTH", "STD_DEPTH", "DATA",
    "PPI", "MAX_BARRAGE", "PENALTY"
]

# Ratio/proportion features
RATIO_FEATURES = [
    "REPEATED_REQUESTS", "NIGHT", "UNASSIGNED",
    "HTML_TO_IMAGE", "HTML_TO_CSS", "HTML_TO_JS",
    "CONSECUTIVE", "SF_REFERRER", "SF_FILETYPE"
]

def generate_realistic_sessions(source_df, robot_label, n_sessions=100, seed=42):
    rng = np.random.default_rng(seed)

    source = source_df[source_df["ROBOT"] == robot_label].copy()

    # Sample real behavioral patterns
    sampled = source.sample(
        n=n_sessions,
        replace=True,
        random_state=seed
    ).copy()

    simulated = sampled[SIM_FEATURES].copy()

    for col in SIM_FEATURES:

        values = pd.to_numeric(simulated[col], errors="coerce")

        # Fill missing values using the class median
        median_value = pd.to_numeric(
            source[col], errors="coerce"
        ).median()

        values = values.fillna(median_value)

        # Small controlled perturbation
        if values.std() > 0:
            noise = rng.normal(
                0,
                values.std() * 0.04,
                size=len(values)
            )
            values = values + noise

        # Physical validity
        if col in NON_NEGATIVE:
            values = values.clip(lower=0)

        # Ratios must stay between 0 and 1
        if col in RATIO_FEATURES:
            values = values.clip(lower=0, upper=1)

        simulated[col] = values

    simulated.insert(
        0,
        "SIM_SESSION_ID",
        [f"sim_{robot_label}_{i:04d}" for i in range(n_sessions)]
    )

    simulated["TRUE_LABEL"] = robot_label

    return simulated


# Generate 100 human + 100 bot sessions
sim_humans = generate_realistic_sessions(
    df, 0, n_sessions=100, seed=42
)

sim_bots = generate_realistic_sessions(
    df, 1, n_sessions=100, seed=99
)

simulator_data = pd.concat(
    [sim_humans, sim_bots],
    ignore_index=True
)

print("Simulator shape:", simulator_data.shape)
print("\nLabels:")
print(simulator_data["TRUE_LABEL"].value_counts())

print("\nMinimum values:")
print(
    simulator_data[SIM_FEATURES]
    .min()
    .sort_values()
    .head(10)
)

display(simulator_data.head())

Simulator shape: (200, 32)

Labels:
TRUE_LABEL
0    100
1    100
Name: count, dtype: int64

Minimum values:
NUMBER_OF_REQUESTS    0.0
TOTAL_DURATION        0.0
AVERAGE_TIME          0.0
STANDARD_DEVIATION    0.0
REPEATED_REQUESTS     0.0
HTTP_RESPONSE_2XX     0.0
HTTP_RESPONSE_3XX     0.0
HTTP_RESPONSE_4XX     0.0
HTTP_RESPONSE_5XX     0.0
GET_METHOD            0.0
dtype: float64


,SIM_SESSION_ID,NUMBER_OF_REQUESTS,TOTAL_DURATION,AVERAGE_TIME,STANDARD_DEVIATION,REPEATED_REQUESTS,HTTP_RESPONSE_2XX,HTTP_RESPONSE_3XX,HTTP_RESPONSE_4XX,HTTP_RESPONSE_5XX,...,DEPTH,STD_DEPTH,CONSECUTIVE,DATA,PPI,SF_REFERRER,SF_FILETYPE,MAX_BARRAGE,PENALTY,TRUE_LABEL
0,sim_0_0000,29.882663,51.523586,3.812202,7.328575,0.206233,1.002392,0.000899,0.000144,0.0,...,5.973474,1.227336,0.381754,145993.940773,0.000000,0.388154,0.540723,7.577918,4.133679,0
1,sim_0_0001,12.987514,1719.557495,119.277921,220.374723,0.250727,1.001570,0.000000,0.000000,0.0,...,6.051242,0.859338,0.233674,52676.639856,0.000000,0.610336,0.602254,2.095459,5.826580,0
2,sim_0_0002,75.173806,12.072345,0.730036,3.079255,0.016729,0.998738,0.002223,0.000000,0.0,...,6.028839,1.326009,0.637038,254918.580026,0.000000,0.127965,0.217468,28.931822,1.130274,0
3,sim_0_0003,4.724501,20.481478,5.116224,0.000000,0.000000,0.997365,0.000000,0.000000,0.0,...,2.993781,0.005248,0.007350,12948.422194,0.000000,0.992645,1.000000,0.158893,0.184975,0
4,sim_0_0004,24.348504,0.000000,0.000000,0.136189,0.001925,0.994800,0.000000,0.000136,0.0,...,6.029613,0.898587,0.364004,225787.864142,26226.589252,0.163283,0.286890,0.915100,0.032884,0


In [29]:
# STEP 12 — Model 1 on 200 simulated sessions

model1_features = list(model1.feature_names_in_)

X_sim = simulator_data[model1_features].copy()

X_sim = X_sim.apply(pd.to_numeric, errors="coerce")

X_sim = X_sim.fillna(
    df[model1_features]
    .apply(pd.to_numeric, errors="coerce")
    .median()
)

sim_predictions = model1.predict(X_sim)

bot_class_index = list(model1.classes_).index(1)

sim_bot_probability = model1.predict_proba(X_sim)[:, bot_class_index]

simulation_results = simulator_data[
    ["SIM_SESSION_ID", "TRUE_LABEL"]
].copy()

simulation_results["MODEL1_PREDICTION"] = sim_predictions
simulation_results["MODEL1_BOT_PROBABILITY"] = sim_bot_probability

simulation_results["TRUE_TYPE"] = simulation_results[
    "TRUE_LABEL"
].map({0: "HUMAN", 1: "BOT"})

simulation_results["MODEL1_TYPE"] = simulation_results[
    "MODEL1_PREDICTION"
].map({0: "HUMAN", 1: "BOT"})

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = simulation_results["TRUE_LABEL"]
y_pred = simulation_results["MODEL1_PREDICTION"]

print("MODEL 1 — SIMULATOR RESULTS")
print("=" * 40)

print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_true, y_pred):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

display(simulation_results.head(20))

MODEL 1 — SIMULATOR RESULTS
Accuracy : 0.9300
Precision: 0.9778
Recall   : 0.8800
F1 Score : 0.9263

Confusion Matrix:
[[98  2]
 [12 88]]


,SIM_SESSION_ID,TRUE_LABEL,MODEL1_PREDICTION,MODEL1_BOT_PROBABILITY,TRUE_TYPE,MODEL1_TYPE
0,sim_0_0000,0,0,0.044000,HUMAN,HUMAN
1,sim_0_0001,0,0,0.048000,HUMAN,HUMAN
2,sim_0_0002,0,0,0.066000,HUMAN,HUMAN
3,sim_0_0003,0,0,0.056009,HUMAN,HUMAN
4,sim_0_0004,0,0,0.020000,HUMAN,HUMAN
5,sim_0_0005,0,0,0.038000,HUMAN,HUMAN
6,sim_0_0006,0,0,0.018000,HUMAN,HUMAN
7,sim_0_0007,0,0,0.014000,HUMAN,HUMAN
8,sim_0_0008,0,0,0.016000,HUMAN,HUMAN
9,sim_0_0009,0,0,0.020000,HUMAN,HUMAN


In [30]:
# STEP 13 — Model 2 anomaly detection on the same 200 sessions

model2_features = list(model2.feature_names_in_)

X_sim_m2 = simulator_data[model2_features].copy()

X_sim_m2 = X_sim_m2.apply(pd.to_numeric, errors="coerce")

X_sim_m2 = X_sim_m2.fillna(
    df[model2_features]
    .apply(pd.to_numeric, errors="coerce")
    .median()
)

# IMPORTANT:
# Use the SAME raw score scale used when the Model 2 threshold was created.
anomaly_scores = -model2.score_samples(X_sim_m2)

MODEL2_THRESHOLD = 0.4830743793999816

model2_predictions = (
    anomaly_scores >= MODEL2_THRESHOLD
).astype(int)

simulation_results["MODEL2_ANOMALY_SCORE"] = anomaly_scores

simulation_results["MODEL2_ANOMALY"] = model2_predictions

simulation_results["MODEL2_TYPE"] = simulation_results[
    "MODEL2_ANOMALY"
].map({
    0: "NORMAL",
    1: "ANOMALOUS"
})

print("MODEL 2 — SIMULATOR RESULTS")
print("=" * 40)

print(
    "Threshold:",
    round(MODEL2_THRESHOLD, 6)
)

print(
    "\nHumans flagged:",
    int(
        simulation_results.loc[
            simulation_results["TRUE_LABEL"] == 0,
            "MODEL2_ANOMALY"
        ].sum()
    ),
    "/ 100"
)

print(
    "Bots flagged:",
    int(
        simulation_results.loc[
            simulation_results["TRUE_LABEL"] == 1,
            "MODEL2_ANOMALY"
        ].sum()
    ),
    "/ 100"
)

display(
    simulation_results[
        [
            "SIM_SESSION_ID",
            "TRUE_TYPE",
            "MODEL1_TYPE",
            "MODEL1_BOT_PROBABILITY",
            "MODEL2_ANOMALY_SCORE",
            "MODEL2_TYPE"
        ]
    ].head(20)
)

MODEL 2 — SIMULATOR RESULTS
Threshold: 0.483074

Humans flagged: 3 / 100
Bots flagged: 37 / 100


,SIM_SESSION_ID,TRUE_TYPE,MODEL1_TYPE,MODEL1_BOT_PROBABILITY,MODEL2_ANOMALY_SCORE,MODEL2_TYPE
0,sim_0_0000,HUMAN,HUMAN,0.044000,0.388309,NORMAL
1,sim_0_0001,HUMAN,HUMAN,0.048000,0.471375,NORMAL
2,sim_0_0002,HUMAN,HUMAN,0.066000,0.387180,NORMAL
3,sim_0_0003,HUMAN,HUMAN,0.056009,0.423541,NORMAL
4,sim_0_0004,HUMAN,HUMAN,0.020000,0.358768,NORMAL
5,sim_0_0005,HUMAN,HUMAN,0.038000,0.360249,NORMAL
6,sim_0_0006,HUMAN,HUMAN,0.018000,0.388234,NORMAL
7,sim_0_0007,HUMAN,HUMAN,0.014000,0.354650,NORMAL
8,sim_0_0008,HUMAN,HUMAN,0.016000,0.422784,NORMAL
9,sim_0_0009,HUMAN,HUMAN,0.020000,0.383029,NORMAL


In [31]:
# STEP 14 — Model 1 vs Model 2 overlap

comparison = simulation_results.copy()

comparison["DETECTION_GROUP"] = "NEITHER"

comparison.loc[
    (comparison["MODEL1_PREDICTION"] == 1) &
    (comparison["MODEL2_ANOMALY"] == 0),
    "DETECTION_GROUP"
] = "MODEL 1 ONLY"

comparison.loc[
    (comparison["MODEL1_PREDICTION"] == 0) &
    (comparison["MODEL2_ANOMALY"] == 1),
    "DETECTION_GROUP"
] = "MODEL 2 ONLY"

comparison.loc[
    (comparison["MODEL1_PREDICTION"] == 1) &
    (comparison["MODEL2_ANOMALY"] == 1),
    "DETECTION_GROUP"
] = "BOTH"

print("DETECTION OVERLAP")
print("=" * 40)

print(
    comparison["DETECTION_GROUP"]
    .value_counts()
)

print("\nBy true type:")
display(
    pd.crosstab(
        comparison["TRUE_TYPE"],
        comparison["DETECTION_GROUP"]
    )
)

DETECTION OVERLAP
DETECTION_GROUP
NEITHER         107
MODEL 1 ONLY     53
BOTH             37
MODEL 2 ONLY      3
Name: count, dtype: int64

By true type:


DETECTION_GROUP,BOTH,MODEL 1 ONLY,MODEL 2 ONLY,NEITHER
TRUE_TYPE,,,,
BOT,37,51,0,12
HUMAN,0,2,3,95


In [32]:
# STEP 15 — Combined Model 1 + Model 2 signal

combined = simulation_results.copy()

combined["COMBINED_SUSPICION"] = (
    0.7 * combined["MODEL1_BOT_PROBABILITY"]
    + 0.3 * combined["MODEL2_ANOMALY"]
)

combined["COMBINED_DECISION"] = (
    combined["COMBINED_SUSPICION"] >= 0.5
).astype(int)

print("COMBINED MODEL RESULTS")
print("=" * 40)

print(
    "Suspicious users:",
    combined["COMBINED_DECISION"].sum(),
    "/",
    len(combined)
)

print("\nBy actual type:")
print(
    pd.crosstab(
        combined["TRUE_TYPE"],
        combined["COMBINED_DECISION"]
    )
)

display(
    combined[
        [
            "SIM_SESSION_ID",
            "TRUE_TYPE",
            "MODEL1_BOT_PROBABILITY",
            "MODEL2_ANOMALY",
            "COMBINED_SUSPICION",
            "COMBINED_DECISION"
        ]
    ].head(20)
)

COMBINED MODEL RESULTS
Suspicious users: 62 / 200

By actual type:
COMBINED_DECISION   0   1
TRUE_TYPE                
BOT                40  60
HUMAN              98   2


,SIM_SESSION_ID,TRUE_TYPE,MODEL1_BOT_PROBABILITY,MODEL2_ANOMALY,COMBINED_SUSPICION,COMBINED_DECISION
0,sim_0_0000,HUMAN,0.044000,0,0.030800,0
1,sim_0_0001,HUMAN,0.048000,0,0.033600,0
2,sim_0_0002,HUMAN,0.066000,0,0.046200,0
3,sim_0_0003,HUMAN,0.056009,0,0.039207,0
4,sim_0_0004,HUMAN,0.020000,0,0.014000,0
5,sim_0_0005,HUMAN,0.038000,0,0.026600,0
6,sim_0_0006,HUMAN,0.018000,0,0.012600,0
7,sim_0_0007,HUMAN,0.014000,0,0.009800,0
8,sim_0_0008,HUMAN,0.016000,0,0.011200,0
9,sim_0_0009,HUMAN,0.020000,0,0.014000,0


In [33]:
# STEP 16 — Inspect the event data for campaign detection

events = pd.read_csv("../repo/data/events.csv")

print("EVENT DATA")
print("=" * 40)

print("Shape:", events.shape)

print("\nColumns:")
print(events.columns.tolist())

print("\nEvent types:")
print(events["event_type"].value_counts())

print("\nActor types:")
print(events["actor_type"].value_counts())

print("\nSessions:")
print(events["session_id"].nunique())

display(events.head(10))

EVENT DATA
Shape: (10192, 7)

Columns:
['session_id', 'actor_type', 'bot_type', 'event_type', 'x', 'y', 'timestamp']

Event types:
event_type
scroll                4125
keydown               3994
mousemove             1181
click                  457
page_screen_change     241
focus_change           182
Name: count, dtype: int64

Actor types:
actor_type
human    5460
bot      4720
Name: count, dtype: int64

Sessions:
83


,session_id,actor_type,bot_type,event_type,x,y,timestamp
0,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,page_screen_change,NaN,NaN,1.774697e+12
1,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,456.0,618.0,1.774697e+12
2,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,497.0,580.0,1.774697e+12
3,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,545.0,516.0,1.774697e+12
4,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,547.0,510.0,1.774697e+12
5,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,866.0,253.0,1.774697e+12
6,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,1007.0,1.0,1.774697e+12
7,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,968.0,47.0,1.774697e+12
8,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,mousemove,1003.0,42.0,1.774697e+12
9,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,click,1003.0,42.0,1.774697e+12


In [34]:
# STEP 17 — Understand the different user types

session_summary = (
    events.groupby("session_id")
    .agg(
        actor_type=("actor_type", "first"),
        bot_type=("bot_type", "first"),
        event_count=("event_type", "count"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max")
    )
    .reset_index()
)

session_summary["duration_ms"] = (
    session_summary["end_time"]
    - session_summary["start_time"]
)

print("SESSION SUMMARY")
print("=" * 40)

print("Total sessions:", len(session_summary))

print("\nHuman / Bot:")
print(session_summary["actor_type"].value_counts())

print("\nBot types:")
print(
    session_summary[
        session_summary["actor_type"] == "bot"
    ]["bot_type"].value_counts()
)

display(session_summary.head(20))

SESSION SUMMARY
Total sessions: 83

Human / Bot:
actor_type
bot      39
human    32
Name: count, dtype: int64

Bot types:
bot_type
coordinated    16
fast           16
human_like      7
Name: count, dtype: int64


,session_id,actor_type,bot_type,event_count,start_time,end_time,duration_ms
0,065f3ad7-9cfa-4737-9ab6-548ecca858fb,human,none,319,1.774720e+12,1.774721e+12,225409.0
1,15e22d8d-1a6c-4693-aed4-914fe34fabe7,human,none,104,1.777522e+12,1.777524e+12,2019512.0
2,187f9d32-2e78-45ff-9ada-818e68545ba3,bot,coordinated,1,1.777621e+12,1.777621e+12,0.0
3,1bed4691-cf7f-4424-a723-d09e9d7c41bb,bot,human_like,164,1.777535e+12,1.777535e+12,20803.0
4,260bd1ca-2fb0-4d82-8908-7bf91c845d8c,bot,human_like,4,1.777524e+12,1.777524e+12,1945.0
5,2871cfdf-3685-43da-a6cb-b80bceca47df,human,none,139,1.777622e+12,1.777622e+12,69514.0
6,2895b8df-3368-49e4-b339-0d6019bc918d,human,none,45,1.777528e+12,1.777528e+12,1059.0
7,2ced526c-35c6-4f0e-9353-6d9a3ee06597,human,none,3,1.777524e+12,1.777524e+12,533.0
8,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,175,1.777535e+12,1.777535e+12,5449.0
9,31ad071d-72eb-4f3d-a95f-99cda830b796,bot,fast,158,1.777620e+12,1.777620e+12,3401.0


In [35]:
# STEP 18 — Find sessions that started close together

usable_sessions = session_summary[
    session_summary["actor_type"].isin(["human", "bot"])
].copy()

usable_sessions = usable_sessions.sort_values(
    "start_time"
).reset_index(drop=True)

usable_sessions["start_time_seconds"] = (
    usable_sessions["start_time"] / 1000
)

print("Usable sessions:", len(usable_sessions))

print("\nEarliest sessions:")
display(
    usable_sessions[
        [
            "session_id",
            "actor_type",
            "bot_type",
            "start_time_seconds"
        ]
    ].head(20)
)

Usable sessions: 71

Earliest sessions:


,session_id,actor_type,bot_type,start_time_seconds
0,6e21434f-90fe-4fbf-b5c7-a8619f1ae73f,human,none,1.774697e+09
1,cccdf178-e77b-4c6b-a237-3fbb798d69c1,bot,fast,1.774697e+09
2,4770ca28-0102-41f8-83c9-bc0b8c2f9ac0,bot,fast,1.774697e+09
3,391534ed-aaec-490c-b62b-bf15d43748a0,bot,human_like,1.774697e+09
4,ee4645c5-afc3-424d-a625-d7fb9f2013ba,human,none,1.774697e+09
5,75b456da-80e3-4b65-852b-b7ae3f44c685,human,none,1.774698e+09
6,7f00c834-d1f3-4978-a81c-0a11723a851d,bot,coordinated,1.774698e+09
7,a77e1e7d-9b64-45d4-8c57-da2a1dbd74ae,bot,coordinated,1.774698e+09
8,fc1551f6-ef6e-496d-b06e-4b71d16b3733,bot,coordinated,1.774698e+09
9,f9a001c7-5c0e-45c4-b881-58940775466b,human,none,1.774698e+09


In [36]:
# STEP 19 — Find sessions that started within 5 seconds of each other

sessions_for_check = usable_sessions[
    ["session_id", "actor_type", "bot_type", "start_time_seconds"]
].copy()

groups = []

for i in range(len(sessions_for_check)):
    current = sessions_for_check.iloc[i]

    nearby = sessions_for_check[
        (sessions_for_check["session_id"] != current["session_id"]) &
        (
            abs(
                sessions_for_check["start_time_seconds"]
                - current["start_time_seconds"]
            ) <= 5
        )
    ].copy()

    for _, other in nearby.iterrows():

        # Avoid recording the same pair twice
        if current["session_id"] < other["session_id"]:

            groups.append({
                "session_1": current["session_id"],
                "type_1": current["actor_type"],
                "bot_type_1": current["bot_type"],
                "session_2": other["session_id"],
                "type_2": other["actor_type"],
                "bot_type_2": other["bot_type"],
                "time_gap_seconds": abs(
                    current["start_time_seconds"]
                    - other["start_time_seconds"]
                )
            })

close_pairs = pd.DataFrame(groups)

print("SESSION PAIRS STARTING WITHIN 5 SECONDS")
print("=" * 50)

print("Number of close pairs:", len(close_pairs))

display(
    close_pairs.sort_values("time_gap_seconds").head(20)
)

SESSION PAIRS STARTING WITHIN 5 SECONDS
Number of close pairs: 41


,session_1,type_1,bot_type_1,session_2,type_2,bot_type_2,time_gap_seconds
0,7f00c834-d1f3-4978-a81c-0a11723a851d,bot,coordinated,a77e1e7d-9b64-45d4-8c57-da2a1dbd74ae,bot,coordinated,0.007
5,67e3357e-bfb2-40de-8fdd-bedb681d0b44,bot,fast,765586f1-b246-4f05-a6a2-cce67a342ed3,bot,human_like,0.027
8,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,5acad450-a957-4338-ac91-acaa2a042172,human,none,0.031
9,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,6cc6bb92-7efe-422c-a18c-a8db3007ddac,human,none,0.039
7,5acad450-a957-4338-ac91-acaa2a042172,human,none,6cc6bb92-7efe-422c-a18c-a8db3007ddac,human,none,0.070
39,4737d77e-5c55-47d8-aa2e-4ea86c2e38fb,human,none,5c0031c4-4542-405a-b26d-a15b224ca26a,human,none,0.071
28,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,ffd821a6-07ba-4dd2-99dc-3281a8e368fb,bot,coordinated,0.180
31,683caf31-c38f-4312-8b0c-2db014a39a4a,bot,coordinated,b91e067e-e91d-49b9-9504-e8770af699ea,bot,coordinated,0.180
30,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,0.180
17,65e023b1-f22f-4b9f-8788-8f873bd0c7ac,bot,coordinated,f4b8657a-27d1-4afb-b8c9-a5045f9bbd16,bot,coordinated,0.180


In [37]:
# STEP 20 — Compare behaviour of close sessions

# Create a simple behaviour summary for each session
behavior = pd.crosstab(
    events["session_id"],
    events["event_type"]
)

# Convert counts into proportions
behavior = behavior.div(
    behavior.sum(axis=1),
    axis=0
).fillna(0)

behavior = behavior.reset_index()

# Add session information
behavior = behavior.merge(
    usable_sessions[
        ["session_id", "actor_type", "bot_type"]
    ],
    on="session_id",
    how="left"
)

print("Behaviour summary shape:", behavior.shape)

display(behavior.head(10))

Behaviour summary shape: (71, 9)


,session_id,click,focus_change,keydown,mousemove,page_screen_change,scroll,actor_type,bot_type
0,065f3ad7-9cfa-4737-9ab6-548ecca858fb,0.012539,0.021944,0.000000,0.225705,0.015674,0.724138,human,none
1,15e22d8d-1a6c-4693-aed4-914fe34fabe7,0.009615,0.125000,0.000000,0.317308,0.009615,0.538462,human,none
2,187f9d32-2e78-45ff-9ada-818e68545ba3,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,bot,coordinated
3,1bed4691-cf7f-4424-a723-d09e9d7c41bb,0.091463,0.006098,0.780488,0.073171,0.030488,0.018293,bot,human_like
4,260bd1ca-2fb0-4d82-8908-7bf91c845d8c,0.000000,0.250000,0.000000,0.500000,0.250000,0.000000,bot,human_like
5,2871cfdf-3685-43da-a6cb-b80bceca47df,0.000000,0.007194,0.000000,0.079137,0.007194,0.906475,human,none
6,2895b8df-3368-49e4-b339-0d6019bc918d,0.000000,0.000000,0.644444,0.000000,0.022222,0.333333,human,none
7,2ced526c-35c6-4f0e-9353-6d9a3ee06597,0.000000,0.000000,0.000000,0.666667,0.333333,0.000000,human,none
8,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,0.085714,0.005714,0.811429,0.051429,0.028571,0.017143,bot,coordinated
9,31ad071d-72eb-4f3d-a95f-99cda830b796,0.094937,0.006329,0.810127,0.037975,0.031646,0.018987,bot,fast


In [38]:
# STEP 21 — Find close users with similar behaviour

from sklearn.metrics.pairwise import cosine_similarity

# Behaviour columns only
behavior_features = [
    "click",
    "focus_change",
    "keydown",
    "mousemove",
    "page_screen_change",
    "scroll"
]

# Calculate similarity between every pair of sessions
similarity_matrix = cosine_similarity(
    behavior[behavior_features]
)

similar_pairs = []

for i in range(len(behavior)):

    for j in range(i + 1, len(behavior)):

        session_a = behavior.iloc[i]
        session_b = behavior.iloc[j]

        # Check whether these two sessions were already found
        # to start within 5 seconds
        pair_match = close_pairs[
            (
                (close_pairs["session_1"] == session_a["session_id"]) &
                (close_pairs["session_2"] == session_b["session_id"])
            )
            |
            (
                (close_pairs["session_1"] == session_b["session_id"]) &
                (close_pairs["session_2"] == session_a["session_id"])
            )
        ]

        if len(pair_match) > 0:

            similarity = similarity_matrix[i, j]

            similar_pairs.append({
                "session_1": session_a["session_id"],
                "type_1": session_a["actor_type"],
                "bot_type_1": session_a["bot_type"],
                "session_2": session_b["session_id"],
                "type_2": session_b["actor_type"],
                "bot_type_2": session_b["bot_type"],
                "time_gap_seconds": pair_match.iloc[0]["time_gap_seconds"],
                "behaviour_similarity": similarity
            })

similar_pairs = pd.DataFrame(similar_pairs)

print("CLOSE + SIMILAR SESSION PAIRS")
print("=" * 50)

print("Pairs found:", len(similar_pairs))

display(
    similar_pairs
    .sort_values(
        ["behaviour_similarity", "time_gap_seconds"],
        ascending=[False, True]
    )
    .head(20)
)

CLOSE + SIMILAR SESSION PAIRS
Pairs found: 41


,session_1,type_1,bot_type_1,session_2,type_2,bot_type_2,time_gap_seconds,behaviour_similarity
36,a77e1e7d-9b64-45d4-8c57-da2a1dbd74ae,bot,coordinated,fc1551f6-ef6e-496d-b06e-4b71d16b3733,bot,coordinated,0.287,1.000000
24,5c0031c4-4542-405a-b26d-a15b224ca26a,human,none,62f117c9-2925-4096-8f09-1a1a2cbed252,human,none,1.917,1.000000
12,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,5acad450-a957-4338-ac91-acaa2a042172,human,none,0.031,1.000000
13,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,6cc6bb92-7efe-422c-a18c-a8db3007ddac,human,none,0.039,1.000000
23,5acad450-a957-4338-ac91-acaa2a042172,human,none,6cc6bb92-7efe-422c-a18c-a8db3007ddac,human,none,0.070,1.000000
28,67e3357e-bfb2-40de-8fdd-bedb681d0b44,bot,fast,765586f1-b246-4f05-a6a2-cce67a342ed3,bot,human_like,0.027,1.000000
9,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,0.180,1.000000
22,59a560e3-c2d2-4c07-9268-a8c53a1eaced,bot,coordinated,b91e067e-e91d-49b9-9504-e8770af699ea,bot,coordinated,0.180,1.000000
29,683caf31-c38f-4312-8b0c-2db014a39a4a,bot,coordinated,b91e067e-e91d-49b9-9504-e8770af699ea,bot,coordinated,0.180,1.000000
21,59a560e3-c2d2-4c07-9268-a8c53a1eaced,bot,coordinated,683caf31-c38f-4312-8b0c-2db014a39a4a,bot,coordinated,0.360,1.000000


In [39]:
# STEP 22 — Combine timing + behaviour + Model 1

# Get Model 1 result for each session
session_model1 = (
    session_summary[
        ["session_id", "actor_type", "bot_type"]
    ]
    .copy()
)

# For this campaign dataset, create a simple bot-likelihood
# based on the known session behaviour model.
#
# We will first use the existing repo session model if available.

print("Checking available model objects...")

print("Model 1 type:", type(model1).__name__)

print("\nCampaign pairs available:", len(similar_pairs))

display(similar_pairs.head(10))

Checking available model objects...
Model 1 type: RandomForestClassifier

Campaign pairs available: 41


,session_1,type_1,bot_type_1,session_2,type_2,bot_type_2,time_gap_seconds,behaviour_similarity
0,187f9d32-2e78-45ff-9ada-818e68545ba3,bot,coordinated,4737d77e-5c55-47d8-aa2e-4ea86c2e38fb,human,none,4.460,0.029788
1,187f9d32-2e78-45ff-9ada-818e68545ba3,bot,coordinated,5c0031c4-4542-405a-b26d-a15b224ca26a,human,none,4.531,0.028940
2,187f9d32-2e78-45ff-9ada-818e68545ba3,bot,coordinated,62f117c9-2925-4096-8f09-1a1a2cbed252,human,none,2.614,0.028940
3,1bed4691-cf7f-4424-a723-d09e9d7c41bb,bot,human_like,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,3.347,0.999480
4,1bed4691-cf7f-4424-a723-d09e9d7c41bb,bot,human_like,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,3.167,0.999480
5,1bed4691-cf7f-4424-a723-d09e9d7c41bb,bot,human_like,923b58c7-1602-4f98-8413-ae6c1a3960aa,bot,coordinated,4.962,0.998949
6,1bed4691-cf7f-4424-a723-d09e9d7c41bb,bot,human_like,bb83cc53-c962-4802-be6c-e4c2e11eb094,bot,fast,3.426,0.998572
7,1bed4691-cf7f-4424-a723-d09e9d7c41bb,bot,human_like,ffd821a6-07ba-4dd2-99dc-3281a8e368fb,bot,coordinated,2.987,0.999405
8,260bd1ca-2fb0-4d82-8908-7bf91c845d8c,bot,human_like,2ced526c-35c6-4f0e-9353-6d9a3ee06597,human,none,2.060,0.912871
9,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,0.180,1.000000


In [40]:
# STEP 23 — Identify the strongest coordination pairs

campaign_candidates = similar_pairs.copy()

# A pair is considered strongly coordinated when:
# - they started within 1 second
# - their behaviour similarity is at least 0.90

campaign_candidates["STRONG_COORDINATION"] = (
    (campaign_candidates["time_gap_seconds"] <= 1.0) &
    (campaign_candidates["behaviour_similarity"] >= 0.90)
)

strong_pairs = campaign_candidates[
    campaign_candidates["STRONG_COORDINATION"]
].copy()

print("STRONG COORDINATION PAIRS")
print("=" * 50)

print("Total close + similar pairs:", len(campaign_candidates))
print("Strong pairs:", len(strong_pairs))

print("\nBy actual pair type:")

strong_pairs["PAIR_TYPE"] = (
    strong_pairs["type_1"]
    + " + "
    + strong_pairs["type_2"]
)

print(
    strong_pairs["PAIR_TYPE"].value_counts()
)

display(
    strong_pairs[
        [
            "session_1",
            "type_1",
            "bot_type_1",
            "session_2",
            "type_2",
            "bot_type_2",
            "time_gap_seconds",
            "behaviour_similarity"
        ]
    ].sort_values(
        "time_gap_seconds"
    )
)

STRONG COORDINATION PAIRS
Total close + similar pairs: 41
Strong pairs: 20

By actual pair type:
PAIR_TYPE
bot + bot        16
human + human     4
Name: count, dtype: int64


,session_1,type_1,bot_type_1,session_2,type_2,bot_type_2,time_gap_seconds,behaviour_similarity
32,7f00c834-d1f3-4978-a81c-0a11723a851d,bot,coordinated,a77e1e7d-9b64-45d4-8c57-da2a1dbd74ae,bot,coordinated,0.007,0.999999
28,67e3357e-bfb2-40de-8fdd-bedb681d0b44,bot,fast,765586f1-b246-4f05-a6a2-cce67a342ed3,bot,human_like,0.027,1.000000
12,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,5acad450-a957-4338-ac91-acaa2a042172,human,none,0.031,1.000000
13,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,6cc6bb92-7efe-422c-a18c-a8db3007ddac,human,none,0.039,1.000000
23,5acad450-a957-4338-ac91-acaa2a042172,human,none,6cc6bb92-7efe-422c-a18c-a8db3007ddac,human,none,0.070,1.000000
14,4737d77e-5c55-47d8-aa2e-4ea86c2e38fb,human,none,5c0031c4-4542-405a-b26d-a15b224ca26a,human,none,0.071,0.999986
22,59a560e3-c2d2-4c07-9268-a8c53a1eaced,bot,coordinated,b91e067e-e91d-49b9-9504-e8770af699ea,bot,coordinated,0.180,1.000000
9,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,0.180,1.000000
29,683caf31-c38f-4312-8b0c-2db014a39a4a,bot,coordinated,b91e067e-e91d-49b9-9504-e8770af699ea,bot,coordinated,0.180,1.000000
27,65e023b1-f22f-4b9f-8788-8f873bd0c7ac,bot,coordinated,f4b8657a-27d1-4afb-b8c9-a5045f9bbd16,bot,coordinated,0.180,0.999911


In [41]:
# STEP 24 — Create Fair Drop RiskEvents

from datetime import datetime, timezone

risk_events = []

# Create an easy lookup for Model 2 results
model2_lookup = {
    row["SIM_SESSION_ID"]: {
        "anomaly_score": row["MODEL2_ANOMALY_SCORE"],
        "anomaly": row["MODEL2_ANOMALY"]
    }
    for _, row in simulation_results.iterrows()
}

# Create one RiskEvent for each strong coordination pair
for _, row in strong_pairs.iterrows():

    session_1 = row["session_1"]
    session_2 = row["session_2"]

    # Find Model 1 results if these sessions exist
    # in our simulator results
    m1_1 = simulation_results[
        simulation_results["SIM_SESSION_ID"] == session_1
    ]

    m1_2 = simulation_results[
        simulation_results["SIM_SESSION_ID"] == session_2
    ]

    # Start with coordination evidence
    evidence = [
        "sessions_started_within_1_second",
        "very_similar_behaviour"
    ]

    risk_events.append({
        "session_id": session_1,
        "event_id": f"risk_{len(risk_events)+1:04d}",
        "risk_score": round(
            min(
                1.0,
                0.5 + (
                    row["behaviour_similarity"] * 0.3
                ) + (
                    max(
                        0,
                        1 - row["time_gap_seconds"]
                    ) * 0.2
                )
            ),
            4
        ),
        "anomaly_score": (
            round(
                float(
                    m1_1.iloc[0]["MODEL2_ANOMALY_SCORE"]
                ),
                4
            )
            if len(m1_1) > 0 else None
        ),
        "coordination_score": round(
            (
                row["behaviour_similarity"]
                *
                max(
                    0,
                    1 - row["time_gap_seconds"]
                )
            ),
            4
        ),
        "campaign_id": f"campaign_{len(risk_events)+1:03d}",
        "attack_type": "possible_coordinated_campaign",
        "evidence": evidence,
        "model_version": "model1_rf_v1 + model2_if_v1 + coordination_v1",
        "timestamp": datetime.now(timezone.utc).isoformat()
    })

risk_events_df = pd.DataFrame(risk_events)

print("RISKEVENTS CREATED")
print("=" * 40)

print("Total RiskEvents:", len(risk_events_df))

display(risk_events_df.head(10))

RISKEVENTS CREATED
Total RiskEvents: 20


,session_id,event_id,risk_score,anomaly_score,coordination_score,campaign_id,attack_type,evidence,model_version,timestamp
0,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,risk_0001,0.9640,None,0.8200,campaign_001,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.789848+00:00
1,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,risk_0002,0.9280,None,0.6400,campaign_002,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.795091+00:00
2,38b3297c-f7f5-42d5-9683-52dbcecddb9d,risk_0003,0.9938,None,0.9690,campaign_003,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.801194+00:00
3,38b3297c-f7f5-42d5-9683-52dbcecddb9d,risk_0004,0.9922,None,0.9610,campaign_004,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.807603+00:00
4,4737d77e-5c55-47d8-aa2e-4ea86c2e38fb,risk_0005,0.9858,None,0.9290,campaign_005,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.822059+00:00
5,5641770d-75ee-4361-8772-69eba8aacd96,risk_0006,0.9640,None,0.8200,campaign_006,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.840349+00:00
6,59a560e3-c2d2-4c07-9268-a8c53a1eaced,risk_0007,0.9280,None,0.6400,campaign_007,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.853544+00:00
7,59a560e3-c2d2-4c07-9268-a8c53a1eaced,risk_0008,0.9640,None,0.8200,campaign_008,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.862811+00:00
8,5acad450-a957-4338-ac91-acaa2a042172,risk_0009,0.9860,None,0.9300,campaign_009,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.873066+00:00
9,65e023b1-f22f-4b9f-8788-8f873bd0c7ac,risk_0010,0.9171,None,0.5858,campaign_010,possible_coordinated_campaign,"[sessions_started_within_1_second, very_simila...",model1_rf_v1 + model2_if_v1 + coordination_v1,2026-10-03T21:38:43.883604+00:00


In [42]:
# STEP 25 — Campaign detection summary

total_strong_pairs = len(strong_pairs)

bot_bot_pairs = (
    (strong_pairs["type_1"] == "bot") &
    (strong_pairs["type_2"] == "bot")
).sum()

human_human_pairs = (
    (strong_pairs["type_1"] == "human") &
    (strong_pairs["type_2"] == "human")
).sum()

bot_human_pairs = total_strong_pairs - bot_bot_pairs - human_human_pairs

print("FAIR DROP CAMPAIGN DETECTION")
print("=" * 45)

print(f"Strong suspicious pairs : {total_strong_pairs}")
print(f"Bot + Bot pairs         : {bot_bot_pairs}")
print(f"Human + Human pairs     : {human_human_pairs}")
print(f"Bot + Human pairs       : {bot_human_pairs}")

if total_strong_pairs > 0:
    print(
        f"\nBot-pair precision      : "
        f"{bot_bot_pairs / total_strong_pairs * 100:.1f}%"
    )

print("\nRiskEvents generated    :", len(risk_events_df))

print("\nInterpretation:")
print(
    "The campaign detector found groups of sessions that started "
    "very close together and behaved very similarly."
)
print(
    "These become RiskEvents for the Fair Drop protection layer."
)

FAIR DROP CAMPAIGN DETECTION
Strong suspicious pairs : 20
Bot + Bot pairs         : 16
Human + Human pairs     : 4
Bot + Human pairs       : 0

Bot-pair precision      : 80.0%

RiskEvents generated    : 20

Interpretation:
The campaign detector found groups of sessions that started very close together and behaved very similarly.
These become RiskEvents for the Fair Drop protection layer.


In [43]:
# STEP 26 — Prepare Fair Drop ML package for Backend + Frontend integration

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

# ---------------------------------------------------------
# 1. Create integration folder
# ---------------------------------------------------------
integration_dir = Path("../integration_package")
integration_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. Save simulated session results
# ---------------------------------------------------------
simulator_results_df = simulation_results.copy()

simulator_results_df.to_csv(
    integration_dir / "simulator_sessions.csv",
    index=False
)

# ---------------------------------------------------------
# 3. Save RiskEvents
# ---------------------------------------------------------
risk_events_export = risk_events_df.copy()

risk_events_export.to_csv(
    integration_dir / "risk_events.csv",
    index=False
)

# Convert NaN/None safely for JSON
risk_events_json = risk_events_export.where(
    pd.notnull(risk_events_export),
    None
).to_dict(orient="records")

sessions_json = simulator_results_df.where(
    pd.notnull(simulator_results_df),
    None
).to_dict(orient="records")

# ---------------------------------------------------------
# 4. Create ONE API-ready payload
# ---------------------------------------------------------
api_payload = {
    "schema_version": "fairdrop_ml_v1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source": "FairDrop Person 2 ML",

    "summary": {
        "simulated_sessions": len(sessions_json),
        "risk_events": len(risk_events_json),
        "campaign_strong_pairs": len(strong_pairs),
        "bot_bot_pairs": int(bot_bot_pairs),
        "human_human_pairs": int(human_human_pairs),
        "bot_human_pairs": int(bot_human_pairs)
    },

    "sessions": sessions_json,

    "risk_events": risk_events_json
}

with open(
    integration_dir / "fairdrop_ml_output.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(api_payload, f, indent=2, default=str)

# ---------------------------------------------------------
# 5. Save a simple integration README
# ---------------------------------------------------------
readme = """
# Fair Drop — Person 2 ML Integration Package

This folder contains the ML outputs required by the Fair Drop backend
and frontend.

## Files

### fairdrop_ml_output.json
Main API-ready file.

Contains:
- simulated sessions
- RiskEvents
- campaign detection summary

### simulator_sessions.csv
Session-level simulator output.

### risk_events.csv
Detected suspicious/campaign RiskEvents.

## Backend Integration

Backend can load:

    fairdrop_ml_output.json

and expose the data through an API such as:

    GET /ml/simulation
    GET /ml/risk-events

The backend should NOT allow the ML output to directly modify
ticket inventory or seat allocation.

ML only provides intelligence/evidence.

## Frontend Integration

Frontend can display:

- total simulated users
- human/bot results
- risk score
- anomaly score
- coordination score
- campaign ID
- attack type
- evidence
- model version

Example RiskEvent fields:

    session_id
    event_id
    risk_score
    anomaly_score
    coordination_score
    campaign_id
    attack_type
    evidence
    model_version
    timestamp

## Important

ML does NOT:
- allocate seats
- change inventory
- decide final admission
- trust client queue position

The deterministic backend policy remains responsible
for the final action.
"""

with open(
    integration_dir / "README_INTEGRATION.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

# ---------------------------------------------------------
# 6. Print package summary
# ---------------------------------------------------------
print("=" * 55)
print("FAIR DROP ML INTEGRATION PACKAGE READY")
print("=" * 55)

print(f"Location          : {integration_dir.resolve()}")
print(f"Simulated sessions: {len(sessions_json)}")
print(f"RiskEvents        : {len(risk_events_json)}")
print(f"Campaign pairs    : {len(strong_pairs)}")

print("\nFiles created:")
for file in sorted(integration_dir.iterdir()):
    print(" -", file.name)

print("\nBackend can use:")
print("  fairdrop_ml_output.json")

print("\nFrontend can consume:")
print("  sessions + risk_events")

print("\nML role:")
print("  Detect suspicious behaviour")
print("  Provide evidence")
print("  Never directly modify ticket inventory")

FAIR DROP ML INTEGRATION PACKAGE READY
Location          : C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\integration_package
Simulated sessions: 200
RiskEvents        : 20
Campaign pairs    : 20

Files created:
 - fairdrop_ml_output.json
 - README_INTEGRATION.md
 - risk_events.csv
 - simulator_sessions.csv

Backend can use:
  fairdrop_ml_output.json

Frontend can consume:
  sessions + risk_events

ML role:
  Detect suspicious behaviour
  Provide evidence
  Never directly modify ticket inventory


In [44]:
# STEP 27 — Create backend-ready FastAPI ML server

from pathlib import Path

api_dir = Path("../integration_package/ml_api")
api_dir.mkdir(parents=True, exist_ok=True)

api_code = r'''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pathlib import Path
import json

# ---------------------------------------------------------
# Fair Drop Person 2 — ML API
# ---------------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_FILE = BASE_DIR / "fairdrop_ml_output.json"

with open(DATA_FILE, "r", encoding="utf-8") as f:
    ML_DATA = json.load(f)

app = FastAPI(
    title="Fair Drop ML API",
    version="1.0.0"
)

# Allow the frontend to call this API during development
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def root():
    return {
        "service": "Fair Drop ML",
        "status": "running",
        "version": "1.0.0"
    }


@app.get("/ml/summary")
def get_summary():
    return ML_DATA["summary"]


@app.get("/ml/simulation")
def get_simulation():
    return {
        "count": len(ML_DATA["sessions"]),
        "sessions": ML_DATA["sessions"]
    }


@app.get("/ml/risk-events")
def get_risk_events():
    return {
        "count": len(ML_DATA["risk_events"]),
        "risk_events": ML_DATA["risk_events"]
    }


@app.get("/ml/session/{session_id}")
def get_session(session_id: str):

    for session in ML_DATA["sessions"]:
        if str(session.get("SIM_SESSION_ID")) == session_id:
            return session

    raise HTTPException(
        status_code=404,
        detail="Session not found"
    )


@app.get("/ml/risk-event/{event_id}")
def get_risk_event(event_id: str):

    for event in ML_DATA["risk_events"]:
        if str(event.get("event_id")) == event_id:
            return event

    raise HTTPException(
        status_code=404,
        detail="RiskEvent not found"
    )
'''

api_file = api_dir / "main.py"

with open(api_file, "w", encoding="utf-8") as f:
    f.write(api_code)

requirements = """fastapi
uvicorn
"""

with open(api_dir / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("=" * 55)
print("FAIR DROP ML API CREATED")
print("=" * 55)

print(f"API file : {api_file.resolve()}")
print(f"Requirements: {(api_dir / 'requirements.txt').resolve()}")

print("\nEndpoints:")
print("  GET /")
print("  GET /ml/summary")
print("  GET /ml/simulation")
print("  GET /ml/risk-events")
print("  GET /ml/session/{session_id}")
print("  GET /ml/risk-event/{event_id}")

print("\nNext:")
print("We will run this API locally and test the endpoints.")

FAIR DROP ML API CREATED
API file : C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\integration_package\ml_api\main.py
Requirements: C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\integration_package\ml_api\requirements.txt

Endpoints:
  GET /
  GET /ml/summary
  GET /ml/simulation
  GET /ml/risk-events
  GET /ml/session/{session_id}
  GET /ml/risk-event/{event_id}

Next:
We will run this API locally and test the endpoints.


In [45]:

# STEP 31 — Create frontend/backend integration contract

from pathlib import Path

integration_dir = Path("../integration_package")

contract = """
# Fair Drop — Person 2 ML API Integration Contract

## Base URL

Local development:

http://127.0.0.1:8001

---

## 1. Health Check

GET /

Purpose:
Check whether the ML service is running.

Example response:

{
  "service": "Fair Drop ML",
  "status": "running",
  "version": "1.0.0"
}

---

## 2. ML Summary

GET /ml/summary

Returns:

{
  "simulated_sessions": 200,
  "risk_events": 20,
  "campaign_strong_pairs": 20,
  "bot_bot_pairs": 16,
  "human_human_pairs": 4,
  "bot_human_pairs": 0
}

Frontend can use this for dashboard cards.

---

## 3. All Simulated Sessions

GET /ml/simulation

Returns:

{
  "count": 200,
  "sessions": [...]
}

Frontend can use this to display:

- total sessions
- human/bot simulation results
- Model 1 output
- Model 2 output
- session-level information

---

## 4. RiskEvents

GET /ml/risk-events

Returns:

{
  "count": 20,
  "risk_events": [...]
}

Each RiskEvent contains:

- session_id
- event_id
- risk_score
- anomaly_score
- coordination_score
- campaign_id
- attack_type
- evidence
- model_version
- timestamp

Frontend can use these for:

- suspicious activity table
- campaign detection panel
- risk monitoring
- evidence/details popup

---

## 5. Single Session

GET /ml/session/{session_id}

Example:

GET /ml/session/sim_1_0001

Returns the session information.

If the session does not exist:

HTTP 404

---

## 6. Single RiskEvent

GET /ml/risk-event/{event_id}

Example:

GET /ml/risk-event/risk_0001

Returns one RiskEvent.

If the event does not exist:

HTTP 404

---

# IMPORTANT ARCHITECTURE RULE

Person 2 ML provides intelligence/evidence.

ML must NOT directly:

- allocate seats
- change ticket inventory
- decide final admission
- trust a client-provided queue position

The backend policy layer decides the final action.

Example:

ML:
"Suspicious coordinated behaviour detected."

Backend policy:
"Require additional verification."

Not:

ML:
"Delete/deny the ticket."

---

# FRONTEND EXAMPLE

JavaScript:

const response = await fetch(
    "http://127.0.0.1:8001/ml/summary"
);

const data = await response.json();

console.log(data);

For RiskEvents:

const response = await fetch(
    "http://127.0.0.1:8001/ml/risk-events"
);

const data = await response.json();

console.log(data.risk_events);

---

# BACKEND INTEGRATION

The backend person can either:

1. Keep this ML API as a separate service, OR
2. Import the ML outputs directly into the main backend.

For the hackathon demo, keeping the ML service separate is recommended.

Flow:

Frontend
   |
   v
Main Backend
   |
   v
Person 2 ML API
   |
   +--> Model 1
   |
   +--> Model 2
   |
   +--> Campaign Detection
   |
   v
RiskEvents
   |
   v
Backend Policy
   |
   v
Admission / Verification / Allocation

---

# CURRENT DEMO RESULTS

Simulation:

200 sessions

Campaign detection:

20 strong suspicious pairs
16 bot + bot
4 human + human
0 bot + human

Bot-pair precision for this specific test:

80%

This is NOT a general production accuracy claim.

---

# LOCAL DEVELOPMENT

Start Person 2 ML API:

python -m uvicorn main:app --reload --port 8001

API:

http://127.0.0.1:8001

---

# HANDOFF FILE

Main data file:

fairdrop_ml_output.json

Other files:

simulator_sessions.csv
risk_events.csv
README_INTEGRATION.md

"""

contract_file = integration_dir / "API_INTEGRATION_CONTRACT.md"

with open(contract_file, "w", encoding="utf-8") as f:
    f.write(contract)

print("=" * 55)
print("FRONTEND / BACKEND INTEGRATION CONTRACT READY")
print("=" * 55)

print(f"Created:")
print(contract_file.resolve())

print("\nGive this file to the backend/frontend teammates.")
print("\nMain API:")
print("http://127.0.0.1:8001")

print("\nEndpoints documented:")
print("  GET /")
print("  GET /ml/summary")
print("  GET /ml/simulation")
print("  GET /ml/risk-events")
print("  GET /ml/session/{session_id}")
print("  GET /ml/risk-event/{event_id}")

FRONTEND / BACKEND INTEGRATION CONTRACT READY
Created:
C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\integration_package\API_INTEGRATION_CONTRACT.md

Give this file to the backend/frontend teammates.

Main API:
http://127.0.0.1:8001

Endpoints documented:
  GET /
  GET /ml/summary
  GET /ml/simulation
  GET /ml/risk-events
  GET /ml/session/{session_id}
  GET /ml/risk-event/{event_id}


In [46]:
# STEP 32 — Clean and finalize campaign RiskEvents

import json
from pathlib import Path
from datetime import datetime, timezone

integration_dir = Path("../integration_package")

# ---------------------------------------------------------
# Build honest campaign RiskEvents
# ---------------------------------------------------------

clean_risk_events = []

for i, (_, row) in enumerate(strong_pairs.iterrows(), start=1):

    # Keep only evidence actually produced by the
    # campaign detector.
    evidence = [
        "sessions_started_within_1_second",
        "very_similar_behaviour"
    ]

    clean_risk_events.append({
        "session_id": row["session_1"],
        "event_id": f"risk_{i:04d}",
        
        # No overall risk score is invented here.
        # Final risk should be decided by the backend policy.
        "risk_score": None,
        
        # Model 2 was not applied to this event dataset.
        "anomaly_score": None,
        
        # This is directly measured by the campaign detector.
        "coordination_score": round(
            float(row["similarity"]), 4
        ),
        
        "campaign_id": f"campaign_{i:03d}",
        "attack_type": "possible_coordinated_campaign",
        
        "evidence": evidence,
        
        # Only the detector actually used here.
        "model_version": "coordination_v1",
        
        "timestamp": datetime.now(timezone.utc).isoformat()
    })

risk_events_df = pd.DataFrame(clean_risk_events)

# ---------------------------------------------------------
# Save cleaned RiskEvents
# ---------------------------------------------------------

risk_events_df.to_csv(
    integration_dir / "risk_events.csv",
    index=False
)

# ---------------------------------------------------------
# Update API-ready JSON
# ---------------------------------------------------------

sessions_json = (
    simulator_results
    .where(pd.notnull(simulator_results), None)
    .to_dict(orient="records")
)

risk_events_json = (
    risk_events_df
    .where(pd.notnull(risk_events_df), None)
    .to_dict(orient="records")
)

api_payload = {
    "schema_version": "fairdrop_ml_v1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source": "FairDrop Person 2 ML",

    "summary": {
        "simulated_sessions": len(sessions_json),
        "risk_events": len(risk_events_json),
        "campaign_strong_pairs": len(strong_pairs),
        "bot_bot_pairs": int(bot_bot_pairs),
        "human_human_pairs": int(human_human_pairs),
        "bot_human_pairs": int(bot_human_pairs)
    },

    "sessions": sessions_json,
    "risk_events": risk_events_json
}

with open(
    integration_dir / "fairdrop_ml_output.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(api_payload, f, indent=2, default=str)

# ---------------------------------------------------------
# Show final result
# ---------------------------------------------------------

print("=" * 55)
print("RISK EVENTS CLEANED + FINALIZED")
print("=" * 55)

print("Total RiskEvents:", len(risk_events_df))

print("\nModel version:")
print(risk_events_df["model_version"].unique().tolist())

print("\nRisk score:")
print("Not invented — left empty until backend policy combines signals.")

print("\nAnomaly score:")
print("Not available for this event dataset.")

print("\nCoordination score:")
print(
    f"{risk_events_df['coordination_score'].min():.4f}"
    " → "
    f"{risk_events_df['coordination_score'].max():.4f}"
)

print("\nUpdated files:")
print(" - risk_events.csv")
print(" - fairdrop_ml_output.json")

print("\nPerson 2 pipeline:")
print("Model 1 → known bot behaviour")
print("Model 2 → unusual behaviour")
print("Campaign detector → coordinated behaviour")
print("RiskEvent → evidence for backend policy")

KeyError: 'similarity'

In [47]:
# STEP 32A — Check campaign pair columns

print("strong_pairs columns:")
print(strong_pairs.columns.tolist())

print("\nFirst 3 rows:")
display(strong_pairs.head(3))

strong_pairs columns:
['session_1', 'type_1', 'bot_type_1', 'session_2', 'type_2', 'bot_type_2', 'time_gap_seconds', 'behaviour_similarity', 'STRONG_COORDINATION', 'PAIR_TYPE']

First 3 rows:


,session_1,type_1,bot_type_1,session_2,type_2,bot_type_2,time_gap_seconds,behaviour_similarity,STRONG_COORDINATION,PAIR_TYPE
9,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,5641770d-75ee-4361-8772-69eba8aacd96,bot,coordinated,0.180,1.00000,True,bot + bot
10,2f2a5540-fceb-4e5d-a52e-c150885a4c2d,bot,coordinated,ffd821a6-07ba-4dd2-99dc-3281a8e368fb,bot,coordinated,0.360,0.99998,True,bot + bot
12,38b3297c-f7f5-42d5-9683-52dbcecddb9d,human,none,5acad450-a957-4338-ac91-acaa2a042172,human,none,0.031,1.00000,True,human + human


In [50]:
# STEP 32D — Safely find likely simulator dataframes

import pandas as pd

names = list(globals().keys())

print("Likely simulator/dataframe variables:\n")

for name in names:
    try:
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            cols = set(obj.columns)

            # Look for simulator-related columns
            if (
                "SIM_SESSION_ID" in cols
                or "MODEL1_BOT_PROBABILITY" in cols
                or "MODEL2_ANOMALY_SCORE" in cols
                or "TRUE_LABEL" in cols
            ):
                print(f"\n{name}")
                print("shape:", obj.shape)
                print("columns:", list(obj.columns))

    except Exception:
        pass

Likely simulator/dataframe variables:


comparison
shape: (200, 10)
columns: ['SIM_SESSION_ID', 'TRUE_LABEL', 'MODEL1_PREDICTION', 'MODEL1_BOT_PROBABILITY', 'TRUE_TYPE', 'MODEL1_TYPE', 'MODEL2_ANOMALY_SCORE', 'MODEL2_ANOMALY', 'MODEL2_TYPE', 'DETECTION_GROUP']

sim_humans
shape: (100, 32)
columns: ['SIM_SESSION_ID', 'NUMBER_OF_REQUESTS', 'TOTAL_DURATION', 'AVERAGE_TIME', 'STANDARD_DEVIATION', 'REPEATED_REQUESTS', 'HTTP_RESPONSE_2XX', 'HTTP_RESPONSE_3XX', 'HTTP_RESPONSE_4XX', 'HTTP_RESPONSE_5XX', 'GET_METHOD', 'POST_METHOD', 'HEAD_METHOD', 'OTHER_METHOD', 'NIGHT', 'UNASSIGNED', 'IMAGES', 'TOTAL_HTML', 'HTML_TO_IMAGE', 'HTML_TO_CSS', 'HTML_TO_JS', 'WIDTH', 'DEPTH', 'STD_DEPTH', 'CONSECUTIVE', 'DATA', 'PPI', 'SF_REFERRER', 'SF_FILETYPE', 'MAX_BARRAGE', 'PENALTY', 'TRUE_LABEL']

sim_bots
shape: (100, 32)
columns: ['SIM_SESSION_ID', 'NUMBER_OF_REQUESTS', 'TOTAL_DURATION', 'AVERAGE_TIME', 'STANDARD_DEVIATION', 'REPEATED_REQUESTS', 'HTTP_RESPONSE_2XX', 'HTTP_RESPONSE_3XX', 'HTTP_RESPONSE_4XX'

In [51]:
# STEP 32E — FINAL RiskEvent cleanup + API JSON rebuild

import json
from pathlib import Path
from datetime import datetime, timezone

integration_dir = Path("../integration_package")

# ---------------------------------------------------------
# 1. Build clean RiskEvents
# ---------------------------------------------------------

clean_risk_events = []

for i, (_, row) in enumerate(strong_pairs.iterrows(), start=1):

    clean_risk_events.append({
        "session_id": row["session_1"],
        "event_id": f"risk_{i:04d}",

        # We do NOT invent an overall risk score.
        "risk_score": None,

        # Model 2 was not applied to these event sessions.
        "anomaly_score": None,

        # Direct campaign evidence.
        "coordination_score": round(
            float(row["behaviour_similarity"]), 4
        ),

        "campaign_id": f"campaign_{i:03d}",

        "attack_type": "possible_coordinated_campaign",

        "evidence": [
            "sessions_started_within_1_second",
            "very_similar_behaviour",
            f"time_gap_seconds={float(row['time_gap_seconds']):.3f}",
            f"behaviour_similarity={float(row['behaviour_similarity']):.4f}"
        ],

        "model_version": "coordination_v1",

        "timestamp": datetime.now(timezone.utc).isoformat()
    })

risk_events_df = pd.DataFrame(clean_risk_events)

# ---------------------------------------------------------
# 2. Save RiskEvents CSV
# ---------------------------------------------------------

risk_events_df.to_csv(
    integration_dir / "risk_events.csv",
    index=False
)

# ---------------------------------------------------------
# 3. Use the CORRECT 200-session simulator dataframe
# ---------------------------------------------------------

sessions_json = (
    simulation_results
    .where(pd.notnull(simulation_results), None)
    .to_dict(orient="records")
)

risk_events_json = (
    risk_events_df
    .where(pd.notnull(risk_events_df), None)
    .to_dict(orient="records")
)

# ---------------------------------------------------------
# 4. Build final API payload
# ---------------------------------------------------------

api_payload = {
    "schema_version": "fairdrop_ml_v1",

    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "FairDrop Person 2 ML",

    "summary": {
        "simulated_sessions": len(sessions_json),
        "risk_events": len(risk_events_json),
        "campaign_strong_pairs": len(strong_pairs),
        "bot_bot_pairs": int(bot_bot_pairs),
        "human_human_pairs": int(human_human_pairs),
        "bot_human_pairs": int(bot_human_pairs)
    },

    "sessions": sessions_json,

    "risk_events": risk_events_json
}

# ---------------------------------------------------------
# 5. Save final JSON
# ---------------------------------------------------------

with open(
    integration_dir / "fairdrop_ml_output.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        api_payload,
        f,
        indent=2,
        default=str
    )

# ---------------------------------------------------------
# 6. Final verification
# ---------------------------------------------------------

print("=" * 60)
print("FAIR DROP PERSON 2 — FINAL ML PACKAGE")
print("=" * 60)

print("\nSimulator sessions :", len(sessions_json))
print("RiskEvents          :", len(risk_events_json))
print("Strong campaign pairs:", len(strong_pairs))

print("\nCampaign result:")
print("  Bot + Bot         :", bot_bot_pairs)
print("  Human + Human     :", human_human_pairs)
print("  Bot + Human       :", bot_human_pairs)

print("\nRiskEvent model:")
print(
    risk_events_df["model_version"]
    .unique()
    .tolist()
)

print("\nCoordination score:")
print(
    round(risk_events_df["coordination_score"].min(), 4),
    "→",
    round(risk_events_df["coordination_score"].max(), 4)
)

print("\nFiles updated:")
print("  ✓ risk_events.csv")
print("  ✓ fairdrop_ml_output.json")

print("\nArchitecture:")
print("  Model 1          → known bot behaviour")
print("  Model 2          → unusual behaviour")
print("  Campaign detector→ coordinated behaviour")
print("  RiskEvent        → evidence for backend policy")

print("\n✓ Person 2 ML output is ready for integration.")

FAIR DROP PERSON 2 — FINAL ML PACKAGE

Simulator sessions : 200
RiskEvents          : 20
Strong campaign pairs: 20

Campaign result:
  Bot + Bot         : 16
  Human + Human     : 4
  Bot + Human       : 0

RiskEvent model:
['coordination_v1']

Coordination score:
0.9982 → 1.0

Files updated:
  ✓ risk_events.csv
  ✓ fairdrop_ml_output.json

Architecture:
  Model 1          → known bot behaviour
  Model 2          → unusual behaviour
  Campaign detector→ coordinated behaviour
  RiskEvent        → evidence for backend policy

✓ Person 2 ML output is ready for integration.


In [52]:
# STEP 33 — Add live simulator endpoint to the Fair Drop ML API

from pathlib import Path

api_file = Path("../integration_package/ml_api/main.py")

# Read current API
api_code = api_file.read_text(encoding="utf-8")

# Don't add it twice
if "/ml/live" not in api_code:

    live_code = r'''

# ---------------------------------------------------------
# LIVE SIMULATOR
# ---------------------------------------------------------

LIVE_INDEX = 0


@app.get("/ml/live")
def get_live_session():

    global LIVE_INDEX

    sessions = ML_DATA["sessions"]

    if not sessions:
        return {
            "status": "empty",
            "session": None
        }

    # Send one session at a time
    session = sessions[LIVE_INDEX]

    # Move to the next session
    LIVE_INDEX = (LIVE_INDEX + 1) % len(sessions)

    return {
        "status": "ok",
        "sequence": LIVE_INDEX,
        "session": session
    }


@app.post("/ml/live/reset")
def reset_live_simulator():

    global LIVE_INDEX

    LIVE_INDEX = 0

    return {
        "status": "reset",
        "message": "Live simulator restarted",
        "next_session": 1
    }

'''

    api_code += live_code

    api_file.write_text(
        api_code,
        encoding="utf-8"
    )

    print("✓ /ml/live endpoint added")
    print("✓ /ml/live/reset endpoint added")

else:
    print("Live endpoints already exist.")

print("\nAPI file:")
print(api_file.resolve())

print("\nNew endpoints:")
print("GET  /ml/live")
print("POST /ml/live/reset")

print("\nFrontend usage:")
print("fetch('http://127.0.0.1:8001/ml/live')")

✓ /ml/live endpoint added
✓ /ml/live/reset endpoint added

API file:
C:\Users\Sohana Pilli\Downloads\FairDrop_Person2_FINAL\FairDrop_Person2_FINAL\integration_package\ml_api\main.py

New endpoints:
GET  /ml/live
POST /ml/live/reset

Frontend usage:
fetch('http://127.0.0.1:8001/ml/live')


In [53]:
# STEP 36 — Add live RiskEvent stream

from pathlib import Path

api_file = Path("../integration_package/ml_api/main.py")

api_code = api_file.read_text(encoding="utf-8")

if "/ml/live-risk-event" not in api_code:

    live_risk_code = r'''

# ---------------------------------------------------------
# LIVE RISK EVENT STREAM
# ---------------------------------------------------------

RISK_EVENT_INDEX = 0


@app.get("/ml/live-risk-event")
def get_live_risk_event():

    global RISK_EVENT_INDEX

    risk_events = ML_DATA["risk_events"]

    if not risk_events:
        return {
            "status": "empty",
            "risk_event": None
        }

    event = risk_events[RISK_EVENT_INDEX]

    RISK_EVENT_INDEX = (
        RISK_EVENT_INDEX + 1
    ) % len(risk_events)

    return {
        "status": "ok",
        "sequence": RISK_EVENT_INDEX,
        "risk_event": event
    }


@app.post("/ml/live-risk-event/reset")
def reset_live_risk_events():

    global RISK_EVENT_INDEX

    RISK_EVENT_INDEX = 0

    return {
        "status": "reset",
        "message": "RiskEvent stream restarted",
        "next_event": 1
    }

'''

    api_code += live_risk_code

    api_file.write_text(
        api_code,
        encoding="utf-8"
    )

    print("✓ Live RiskEvent endpoint added")
    print("✓ RiskEvent reset endpoint added")

else:
    print("Live RiskEvent endpoints already exist.")

print("\nNew endpoints:")
print("GET  /ml/live-risk-event")
print("POST /ml/live-risk-event/reset")

✓ Live RiskEvent endpoint added
✓ RiskEvent reset endpoint added

New endpoints:
GET  /ml/live-risk-event
POST /ml/live-risk-event/reset


In [54]:
# STEP 38 — Create one frontend-friendly demo endpoint

from pathlib import Path

api_file = Path("../integration_package/ml_api/main.py")

api_code = api_file.read_text(encoding="utf-8")

if "/ml/demo" not in api_code:

    demo_code = r'''

# ---------------------------------------------------------
# FRONTEND DEMO ENDPOINT
# ---------------------------------------------------------

@app.get("/ml/demo")
def get_demo():

    global LIVE_INDEX
    global RISK_EVENT_INDEX

    sessions = ML_DATA["sessions"]
    risk_events = ML_DATA["risk_events"]

    # Next simulated session
    if sessions:
        session = sessions[LIVE_INDEX]
        LIVE_INDEX = (LIVE_INDEX + 1) % len(sessions)
    else:
        session = None

    # Next campaign RiskEvent
    if risk_events:
        risk_event = risk_events[RISK_EVENT_INDEX]
        RISK_EVENT_INDEX = (
            RISK_EVENT_INDEX + 1
        ) % len(risk_events)
    else:
        risk_event = None

    return {
        "status": "ok",

        "session": {
            "source": "200_session_simulator",
            "data": session
        },

        "campaign_alert": {
            "source": "campaign_detector",
            "data": risk_event
        },

        "summary": ML_DATA["summary"]
    }

'''

    api_code += demo_code

    api_file.write_text(
        api_code,
        encoding="utf-8"
    )

    print("✓ /ml/demo endpoint added")

else:
    print("/ml/demo already exists.")

print("\nFrontend now has one simple endpoint:")
print("GET /ml/demo")

print("\nIt returns:")
print("  1. simulated session")
print("  2. campaign RiskEvent")
print("  3. overall ML summary")

✓ /ml/demo endpoint added

Frontend now has one simple endpoint:
GET /ml/demo

It returns:
  1. simulated session
  2. campaign RiskEvent
  3. overall ML summary
